# Healthcare Challenge 2 - Baseline Submission

This notebook provides a simple baseline for **Healthcare Challenge 2: ED Cost Prediction**.

**Goal**: Predict `ed_cost_next3y_usd` for each patient
**Metric**: Mean Absolute Error (MAE) - Lower is better

## Instructions:
1. **Replace API credentials** in the first cell with your team's API key and name
2. **Run all cells** to generate and submit baseline predictions
3. **Check the output** for your submission score

This baseline uses only tabular ED cost data with a simple Random Forest regressor.


In [ ]:
# 1. Initialize Client and Load Data

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from agentds import BenchmarkClient

# 🔑 REPLACE WITH YOUR CREDENTIALS
client = BenchmarkClient(
    api_key="your_api_key",        # Get from your team dashboard
    team_name="your_team_name"     # Your exact team name
)

# Iteration 1
- 463.7263

## EDA

In [32]:
# ============================================================
# AgentDS Challenge 2 — Phase 1 EDA (CSV + PDF receipts) — ONE CELL
# Jupyter-friendly: thread parsing + robust cache validation
# ============================================================

import os
import re
import time
import math
import json
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from joblib import dump, load

# -----------------------------
# Optional deps
# -----------------------------
try:
    import fitz  # PyMuPDF
except Exception as e:
    raise RuntimeError("PyMuPDF missing. Install: pip install pymupdf") from e

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):  # fallback
        return x

warnings.filterwarnings("ignore")

# -----------------------------
# USER CONFIG (as provided)
# -----------------------------
BASE_DIR = Path(r"D:\AgentDs\agent_ds_healthcare")
TRAIN_CSV = BASE_DIR / "ed_cost_train.csv"
TEST_CSV  = BASE_DIR / "ed_cost_test.csv"
PATIENTS_CSV = BASE_DIR / "patients.csv"
PDF_DIR = BASE_DIR / "receipts_pdf"
SUBMISSION_PATH = BASE_DIR / "submission_ICHI_V1.csv"

CACHE_DIR = BASE_DIR / "cache_iter10"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# NOTE: if you set MAX_PDFS for a quick run, we automatically write to a separate cache
MAX_PDFS = None          # set to int for quick dry run (e.g., 200). None = parse all.
FORCE_REPARSE = False    # set True to force rebuilding cache once

RECEIPT_CACHE = CACHE_DIR / ("receipts_parsed.joblib" if MAX_PDFS is None else f"receipts_parsed_max{MAX_PDFS}.joblib")

OUT_DIR = CACHE_DIR / "eda_outputs_iter10"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Analysis knobs
N_TOP_CODES = 40
CODE_MIN_SUPPORT = 20
TOTAL_TOL = 0.01

# Threaded parsing (Windows/Jupyter friendly)
N_WORKERS = max(4, (os.cpu_count() or 8) - 2)

# ============================================================
# Helpers
# ============================================================
def _normalize_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    return df

def _ensure_col_alias(df: pd.DataFrame, aliases: dict) -> pd.DataFrame:
    df = df.copy()
    cols = set(df.columns)
    for canon, alts in aliases.items():
        if canon in cols:
            continue
        for a in alts:
            if a in cols:
                df = df.rename(columns={a: canon})
                cols = set(df.columns)
                break
    return df

def print_header(title: str):
    print("\n" + "="*90)
    print(title)
    print("="*90)

def savefig(path: Path):
    plt.tight_layout()
    plt.savefig(path, dpi=140)
    plt.close()

def safe_float(x):
    if x is None:
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    s = str(x).strip()
    s = s.replace("$", "").replace(",", "")
    try:
        return float(s)
    except Exception:
        return np.nan

def safe_int(x):
    try:
        s = str(x).strip()
        if s == "" or s.lower() == "nan":
            return np.nan
        return int(float(s))
    except Exception:
        return np.nan

def spearman_corr(a, b):
    a = pd.Series(a).astype(float)
    b = pd.Series(b).astype(float)
    m = a.notna() & b.notna()
    if m.sum() < 3:
        return np.nan
    return a[m].rank().corr(b[m].rank())

def cache_is_valid(parsed_obj, expected_n: int) -> bool:
    """Validate that cache is dict[int -> dict with required keys], and roughly complete."""
    if not isinstance(parsed_obj, dict):
        return False
    if expected_n is not None:
        if len(parsed_obj) < int(0.98 * expected_n):
            return False
    # spot-check a few values
    checked = 0
    for v in parsed_obj.values():
        checked += 1
        if not isinstance(v, dict):
            return False
        if ("items" not in v) or ("pdf_total" not in v) or ("text_len" not in v):
            return False
        if checked >= 3:
            break
    return True

# ============================================================
# Receipt parsing
# ============================================================
# IMPORTANT: your synthetic ZIP3 can be 1–3 digits (e.g., "21"), so allow 1–3 digits
ZIP3_RE  = re.compile(r"\bzip3\b\s*[:\-]?\s*(\d{1,3})", re.IGNORECASE)
INS_RE   = re.compile(r"\binsurance\b\s*[:\-]?\s*([A-Za-z_]+)", re.IGNORECASE)
TOTAL_RE = re.compile(r"\btotal\b\s*[:\-]?\s*\$?\s*([\d,]+(?:\.\d{2})?)", re.IGNORECASE)

# CPT/HCPCS-ish codes
CODE_RE = re.compile(r"^[A-Z]?\d{4,5}$|^[A-Z0-9]{4,7}$")

def extract_pdf_text(pdf_path: Path) -> str:
    doc = fitz.open(pdf_path)
    try:
        parts = []
        for i in range(doc.page_count):
            page = doc.load_page(i)
            # sort=True improves reading order consistency for many PDFs
            parts.append(page.get_text("text", sort=True))
        return "\n".join(parts)
    finally:
        doc.close()

def parse_line_item(line: str):
    """
    Heuristic parse:
        <CODE> <DESCRIPTION...> <QTY> <AMOUNT>
    Example:
        99283 Emergency Department Visit (Level 3) 1 245.00
    """
    line = line.strip()
    if not line:
        return None

    # Must end with an amount like 123.45
    m_amt = re.search(r"(\d[\d,]*\.\d{2})\s*$", line)
    if not m_amt:
        return None
    amt = safe_float(m_amt.group(1))
    left = line[:m_amt.start()].strip()

    # qty should end the left side
    m_qty = re.search(r"(\d+)\s*$", left)
    if not m_qty:
        return None
    qty = int(m_qty.group(1))
    left2 = left[:m_qty.start()].strip()
    if not left2:
        return None

    code = left2.split()[0].strip()
    if not CODE_RE.match(code):
        return None

    desc = left2[len(code):].strip()
    return {"code": code, "desc": desc, "qty": qty, "amount": float(amt)}

def parse_receipt(pdf_path: Path):
    text = extract_pdf_text(pdf_path)

    zip3 = None
    m = ZIP3_RE.search(text)
    if m:
        zip3 = m.group(1)

    insurance = None
    m = INS_RE.search(text)
    if m:
        insurance = m.group(1).lower()

    total = None
    totals = TOTAL_RE.findall(text)
    if totals:
        total = safe_float(totals[-1])  # use last TOTAL occurrence

    items = []
    for raw in text.splitlines():
        it = parse_line_item(raw)
        if it is not None:
            items.append(it)

    return {
        "pdf_path": str(pdf_path),
        "zip3_receipt": zip3,
        "insurance_receipt": insurance,
        "pdf_total": total,
        "n_items_raw": len(items),
        "items": items,
        "text_len": len(text),
    }

# ============================================================
# Receipt features (phenotypes)
# ============================================================
def em_level_from_code(code: str):
    # ED E/M codes often 99281–99285
    if not isinstance(code, str):
        return None
    if code.isdigit() and code.startswith("9928") and len(code) == 5:
        last = int(code[-1])
        if last in [1,2,3,4,5]:
            return last
    return None

def code_bucket(code: str):
    if not isinstance(code, str) or not code:
        return "unknown"
    if code.isdigit():
        if code.startswith("99"):
            return "em_99xxx"
        return f"num_{code[0]}xxxx"
    return f"alpha_{code[0].upper()}"

def build_receipt_features(parsed_obj):
    # HARDEN: never crash if cache contains unexpected values
    if not isinstance(parsed_obj, dict):
        return {
            "receipt_parse_failed": 1,
            "receipt_n_items": 0,
            "receipt_n_unique_codes": 0,
            "receipt_sum_items": 0.0,
            "receipt_top1_share": np.nan,
            "receipt_top3_share": np.nan,
            "receipt_em_count": 0,
            "receipt_em_max": np.nan,
            "receipt_em_mean": np.nan,
            "receipt_non_em_count": 0,
            "receipt_bucket_entropy": np.nan,
            "receipt_zip3_from_pdf": None,
            "receipt_ins_from_pdf": None,
            "receipt_pdf_total": np.nan,
        }

    items = parsed_obj.get("items", [])
    if not items:
        return {
            "receipt_parse_failed": 1,
            "receipt_n_items": 0,
            "receipt_n_unique_codes": 0,
            "receipt_sum_items": 0.0,
            "receipt_top1_share": np.nan,
            "receipt_top3_share": np.nan,
            "receipt_em_count": 0,
            "receipt_em_max": np.nan,
            "receipt_em_mean": np.nan,
            "receipt_non_em_count": 0,
            "receipt_bucket_entropy": np.nan,
            "receipt_zip3_from_pdf": parsed_obj.get("zip3_receipt"),
            "receipt_ins_from_pdf": parsed_obj.get("insurance_receipt"),
            "receipt_pdf_total": parsed_obj.get("pdf_total"),
        }

    amounts = np.array([it.get("amount", np.nan) for it in items], dtype=float)
    codes = [it.get("code", "") for it in items]

    uniq = len(set(codes))
    s = float(np.nansum(amounts))
    top_sorted = np.sort(amounts[~np.isnan(amounts)])[::-1]
    top1 = float(top_sorted[0]) if len(top_sorted) else np.nan
    top3 = float(top_sorted[:3].sum()) if len(top_sorted) >= 3 else float(top_sorted.sum()) if len(top_sorted) else np.nan
    top1_share = (top1 / s) if (s and s > 0 and not np.isnan(top1)) else np.nan
    top3_share = (top3 / s) if (s and s > 0 and not np.isnan(top3)) else np.nan

    em_levels = [em_level_from_code(c) for c in codes]
    em_levels = [x for x in em_levels if x is not None]
    em_count = len(em_levels)
    em_max = float(np.max(em_levels)) if em_levels else np.nan
    em_mean = float(np.mean(em_levels)) if em_levels else np.nan

    non_em_count = sum(1 for c in codes if not (isinstance(c, str) and c.isdigit() and c.startswith("99")))

    buckets = [code_bucket(c) for c in codes]
    bc = Counter(buckets)
    probs = np.array(list(bc.values()), dtype=float)
    probs = probs / probs.sum()
    entropy = float(-(probs * np.log(probs + 1e-12)).sum())

    return {
        "receipt_parse_failed": 0,
        "receipt_n_items": len(items),
        "receipt_n_unique_codes": uniq,
        "receipt_sum_items": s,
        "receipt_top1_share": top1_share,
        "receipt_top3_share": top3_share,
        "receipt_em_count": em_count,
        "receipt_em_max": em_max,
        "receipt_em_mean": em_mean,
        "receipt_non_em_count": non_em_count,
        "receipt_bucket_entropy": entropy,
        "receipt_zip3_from_pdf": parsed_obj.get("zip3_receipt"),
        "receipt_ins_from_pdf": parsed_obj.get("insurance_receipt"),
        "receipt_pdf_total": parsed_obj.get("pdf_total"),
    }

# ============================================================
# Main EDA
# ============================================================
def main():
    print_header("LOAD DATA")
    train = _normalize_cols(pd.read_csv(TRAIN_CSV))
    test  = _normalize_cols(pd.read_csv(TEST_CSV))

    aliases = {
        "patient_id": ["patientid", "id"],
        "primary_chronic": ["chronic", "primarycondition"],
        "prior_ed_visits_5y": ["prior_visits_5y", "ed_visits_5y"],
        "prior_ed_cost_5y_usd": ["prior_cost_5y_usd", "ed_cost_5y", "prior_cost_5y"],
        "ed_cost_next3y_usd": ["target", "y", "cost_next3y"],
        "zip3": ["zip", "zip_3", "zip_prefix", "zip3_code", "zip3"],
        "insurance": ["payer", "insurance_type"],
    }
    train = _ensure_col_alias(train, aliases)
    test  = _ensure_col_alias(test, aliases)

    print("Train shape:", train.shape)
    print("Test  shape:", test.shape)
    print("Train cols:", list(train.columns))

    # patients
    patients = None
    if PATIENTS_CSV.exists():
        patients = _normalize_cols(pd.read_csv(PATIENTS_CSV))
        patients = _ensure_col_alias(patients, aliases)
        print("Patients shape:", patients.shape)
        print("Patients cols:", list(patients.columns))
    else:
        print("WARNING: patients.csv not found at", PATIENTS_CSV)

    assert "patient_id" in train.columns and "patient_id" in test.columns, "Missing patient_id"

    dup_tr = train["patient_id"].duplicated().sum()
    dup_te = test["patient_id"].duplicated().sum()
    print(f"Duplicate patient_id — train: {dup_tr} | test: {dup_te}")

    overlap = set(train["patient_id"]).intersection(set(test["patient_id"]))
    print("Train/Test patient_id overlap:", len(overlap))

    # merge patients
    if patients is not None and "patient_id" in patients.columns:
        train = train.merge(patients, on="patient_id", how="left", suffixes=("", "_patients"))
        test  = test.merge(patients, on="patient_id", how="left", suffixes=("", "_patients"))

    # missingness
    print_header("MISSINGNESS (TOP 30)")
    miss = train.isna().mean().sort_values(ascending=False).head(30)
    print(miss.to_string())

    # target profile
    if "ed_cost_next3y_usd" in train.columns:
        y = train["ed_cost_next3y_usd"].astype(float)
        print_header("TARGET DISTRIBUTION")
        qs = y.quantile([0, .01, .05, .1, .25, .5, .75, .9, .95, .99, 1.0])
        print(qs.to_string())
        print("Mean:", y.mean(), "Std:", y.std(), "Skew:", y.skew())

        # histogram raw
        plt.figure()
        plt.hist(y.clip(upper=y.quantile(0.995)), bins=60)
        plt.title("Target histogram (clipped at 99.5%)")
        plt.xlabel("ed_cost_next3y_usd")
        plt.ylabel("count")
        savefig(OUT_DIR / "target_hist_clipped.png")

        # histogram log
        plt.figure()
        plt.hist(np.log1p(y), bins=60)
        plt.title("log1p(Target) histogram")
        plt.xlabel("log1p(ed_cost_next3y_usd)")
        plt.ylabel("count")
        savefig(OUT_DIR / "target_log_hist.png")

        top = y.quantile(0.95)
        tail_share = y[y >= top].sum() / y.sum()
        print(f"Top 5% contribute ~{tail_share:.3f} of total target spend")

    # chronic summary
    if "primary_chronic" in train.columns and "ed_cost_next3y_usd" in train.columns:
        print_header("TARGET BY PRIMARY_CHRONIC")
        g = train.groupby("primary_chronic")["ed_cost_next3y_usd"]
        summary = g.agg(["count", "mean", "median"])
        summary["p90"] = g.quantile(0.90)
        summary["p95"] = g.quantile(0.95)
        summary["p99"] = g.quantile(0.99)
        print(summary.sort_values("mean", ascending=False).to_string())

        plt.figure(figsize=(8,4))
        cats = list(train["primary_chronic"].dropna().unique())
        data = [np.log1p(train.loc[train["primary_chronic"]==c, "ed_cost_next3y_usd"].astype(float)) for c in cats]
        plt.boxplot(data, labels=cats, showfliers=False)
        plt.title("log1p(Target) by primary_chronic (fliers hidden)")
        plt.ylabel("log1p(ed_cost_next3y_usd)")
        savefig(OUT_DIR / "target_by_chronic_box_log.png")

    # basic relationships
    if "ed_cost_next3y_usd" in train.columns:
        print_header("BASIC RELATIONSHIPS (SCATTERS)")
        if "prior_ed_cost_5y_usd" in train.columns:
            x = train["prior_ed_cost_5y_usd"].astype(float)
            y = train["ed_cost_next3y_usd"].astype(float)
            plt.figure()
            plt.scatter(np.log1p(x), np.log1p(y), s=10, alpha=0.35)
            plt.title("log1p(prior_ed_cost_5y_usd) vs log1p(target)")
            plt.xlabel("log1p(prior_ed_cost_5y_usd)")
            plt.ylabel("log1p(ed_cost_next3y_usd)")
            savefig(OUT_DIR / "scatter_log_priorcost_vs_log_target.png")

        if "prior_ed_visits_5y" in train.columns:
            x = train["prior_ed_visits_5y"].astype(float)
            y = train["ed_cost_next3y_usd"].astype(float)
            plt.figure()
            plt.scatter(x, np.log1p(y), s=10, alpha=0.35)
            plt.title("prior_ed_visits_5y vs log1p(target)")
            plt.xlabel("prior_ed_visits_5y")
            plt.ylabel("log1p(ed_cost_next3y_usd)")
            savefig(OUT_DIR / "scatter_visits_vs_log_target.png")

        if "prior_ed_cost_5y_usd" in train.columns and "prior_ed_visits_5y" in train.columns:
            cpv = train["prior_ed_cost_5y_usd"].astype(float) / (train["prior_ed_visits_5y"].astype(float) + 1e-6)
            plt.figure()
            plt.hist(np.log1p(cpv), bins=60)
            plt.title("log1p(prior cost per visit) distribution")
            plt.xlabel("log1p(prior_ed_cost_5y_usd / prior_ed_visits_5y)")
            plt.ylabel("count")
            savefig(OUT_DIR / "prior_cost_per_visit_log_hist.png")

    # numeric correlations
    if "ed_cost_next3y_usd" in train.columns:
        print_header("SPEARMAN CORRELATIONS (NUMERIC FEATURES)")
        y = train["ed_cost_next3y_usd"].astype(float)
        num_cols = [c for c in train.columns if c != "ed_cost_next3y_usd" and pd.api.types.is_numeric_dtype(train[c])]
        rows = [(c, spearman_corr(train[c], y)) for c in num_cols]
        corr_df = pd.DataFrame(rows, columns=["feature", "spearman_r"]).sort_values("spearman_r", ascending=False)
        print(corr_df.head(20).to_string(index=False))
        corr_df.to_csv(OUT_DIR / "spearman_numeric_features.csv", index=False)

    # ============================================================
    # RECEIPTS: coverage + parse + validation
    # ============================================================
    print_header("RECEIPT PDF COVERAGE CHECK")
    assert PDF_DIR.exists(), f"PDF_DIR not found: {PDF_DIR}"

    pdf_files = sorted(PDF_DIR.glob("receipt_*.pdf"))
    print("PDF files found:", len(pdf_files), "in", PDF_DIR)

    pid_to_pdf = {}
    bad_names = []
    for p in pdf_files:
        m = re.search(r"receipt_(\d+)\.pdf$", p.name)
        if not m:
            bad_names.append(p.name)
            continue
        pid_to_pdf[int(m.group(1))] = p

    all_ids = pd.concat([train["patient_id"], test["patient_id"]], axis=0).astype(int).unique().tolist()
    missing_pdfs = [pid for pid in all_ids if pid not in pid_to_pdf]
    extra_pdfs = [pid for pid in pid_to_pdf.keys() if pid not in set(map(int, all_ids))]

    print("Missing PDFs for patient_ids:", len(missing_pdfs))
    print("Extra PDFs not in train/test:", len(extra_pdfs))
    (OUT_DIR / "missing_pdfs.txt").write_text("\n".join(map(str, missing_pdfs[:5000])))
    (OUT_DIR / "extra_pdfs.txt").write_text("\n".join(map(str, extra_pdfs[:5000])))

    # Parse receipts with cache validation
    expected_n = len(all_ids) if MAX_PDFS is None else min(MAX_PDFS, len(all_ids))

    parsed = None
    if RECEIPT_CACHE.exists() and (not FORCE_REPARSE):
        print("Loading receipt cache:", RECEIPT_CACHE)
        parsed = load(RECEIPT_CACHE)
        print("Cached parsed receipts:", len(parsed) if hasattr(parsed, "__len__") else type(parsed))

        if not cache_is_valid(parsed, expected_n if MAX_PDFS is None else None):
            # If MAX_PDFS is set, we don't enforce size; we enforce structure only
            if MAX_PDFS is not None:
                ok_struct = cache_is_valid(parsed, None)
            else:
                ok_struct = False
            if not ok_struct:
                print("⚠️ Receipt cache INVALID (wrong type/structure/size). Rebuilding cache...")
                backup = RECEIPT_CACHE.with_suffix(f".bad_{int(time.time())}.joblib")
                try:
                    RECEIPT_CACHE.rename(backup)
                    print("Backed up bad cache to:", backup)
                except Exception as e:
                    print("Could not rename bad cache (will overwrite):", e)
                parsed = None

    if parsed is None:
        print("Parsing receipts (threaded)... workers:", N_WORKERS)
        from concurrent.futures import ThreadPoolExecutor, as_completed

        items = [(pid, pid_to_pdf[pid]) for pid in pid_to_pdf.keys()]
        items = sorted(items, key=lambda x: x[0])
        if MAX_PDFS is not None:
            items = items[:MAX_PDFS]

        parsed = {}
        t0 = time.time()
        with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
            futs = {ex.submit(parse_receipt, path): pid for pid, path in items}
            for fut in tqdm(as_completed(futs), total=len(futs), desc="Parsing PDFs"):
                pid = futs[fut]
                try:
                    parsed[pid] = fut.result()
                except Exception as e:
                    parsed[pid] = {
                        "pdf_path": str(pid_to_pdf.get(pid, "")),
                        "zip3_receipt": None,
                        "insurance_receipt": None,
                        "pdf_total": None,
                        "n_items_raw": 0,
                        "items": [],
                        "text_len": 0,
                        "parse_error": repr(e),
                    }

        print(f"Parsed {len(parsed)} receipts in {time.time()-t0:.1f}s")
        dump(parsed, RECEIPT_CACHE)
        print("Saved receipt cache:", RECEIPT_CACHE)

    # Build receipt feature table
    print_header("RECEIPT FEATURE EXTRACTION")
    rec_rows = []
    for pid, obj in parsed.items():
        feats = build_receipt_features(obj)
        feats["patient_id"] = int(pid)
        rec_rows.append(feats)
    rec_feat = pd.DataFrame(rec_rows)
    print("Receipt feature df:", rec_feat.shape)
    rec_feat.to_csv(OUT_DIR / "receipt_features_basic.csv", index=False)

    # Merge into train/test
    train2 = train.merge(rec_feat, on="patient_id", how="left")
    test2  = test.merge(rec_feat, on="patient_id", how="left")

    # Validate TOTAL alignment (train/test)
    if "prior_ed_cost_5y_usd" in train2.columns:
        print_header("PDF TOTAL vs CSV prior_ed_cost_5y_usd VALIDATION")
        for name, df in [("train", train2), ("test", test2)]:
            prior = df["prior_ed_cost_5y_usd"].astype(float)
            total = pd.to_numeric(df["receipt_pdf_total"], errors="coerce")
            diff = (total - prior).abs()

            ok = diff <= TOTAL_TOL
            match_rate = float(ok.mean())
            n_bad = int((~ok).sum())
            print(f"[{name}] match_rate={match_rate:.4f} | mismatches={n_bad}/{len(df)}")

            bad = df.loc[~ok, ["patient_id", "prior_ed_cost_5y_usd", "receipt_pdf_total"]].copy()
            if len(bad):
                bad["abs_diff"] = (pd.to_numeric(bad["receipt_pdf_total"], errors="coerce") - bad["prior_ed_cost_5y_usd"].astype(float)).abs()
                bad = bad.sort_values("abs_diff", ascending=False)
            bad.to_csv(OUT_DIR / f"pdf_total_mismatches_{name}.csv", index=False)

            plt.figure()
            plt.hist(diff.fillna(-1), bins=80)
            plt.title(f"abs(PDF_TOTAL - prior_ed_cost_5y_usd) — {name}")
            plt.xlabel("abs diff (USD), NaN shown as -1")
            plt.ylabel("count")
            savefig(OUT_DIR / f"pdf_total_absdiff_hist_{name}.png")

    # Integrity check: receipt ZIP3 / insurance vs patients.csv (if present)
    if "zip3" in train2.columns and "receipt_zip3_from_pdf" in train2.columns:
        print_header("ZIP3 CONSISTENCY: patients.csv vs receipt PDFs (TRAIN)")
        a = train2["zip3"].apply(safe_int)
        b = train2["receipt_zip3_from_pdf"].apply(safe_int)
        m = (~pd.isna(a)) & (~pd.isna(b))
        if m.sum() > 0:
            match = float((a[m] == b[m]).mean())
            print(f"ZIP3 match rate where both present: {match:.4f} (n={int(m.sum())})")
        else:
            print("No comparable ZIP3 pairs found (receipt parsing may not capture ZIP3).")

    if "insurance" in train2.columns and "receipt_ins_from_pdf" in train2.columns:
        print_header("INSURANCE CONSISTENCY: patients.csv vs receipt PDFs (TRAIN)")
        a = train2["insurance"].astype(str).str.lower().str.strip()
        b = train2["receipt_ins_from_pdf"].astype(str).str.lower().str.strip()
        m = (a != "nan") & (b != "nan")
        if m.sum() > 0:
            match = float((a[m] == b[m]).mean())
            print(f"Insurance match rate where both present: {match:.4f} (n={int(m.sum())})")
        else:
            print("No comparable insurance pairs found (receipt parsing may not capture insurance).")

    # Receipt feature distributions
    print_header("RECEIPT FEATURE SANITY (DISTRIBUTIONS)")
    for c in ["receipt_n_items", "receipt_n_unique_codes", "receipt_em_count", "receipt_em_max",
              "receipt_top1_share", "receipt_bucket_entropy", "receipt_parse_failed"]:
        if c in train2.columns:
            s = pd.to_numeric(train2[c], errors="coerce")
            print(f"{c}: mean={s.mean():.3f} | p50={s.quantile(0.5):.3f} | p90={s.quantile(0.9):.3f} | missing={float(train2[c].isna().mean()):.3f}")

            plt.figure()
            vals = s.dropna()
            if len(vals):
                plt.hist(vals.clip(upper=vals.quantile(0.995)), bins=60)
                plt.title(f"{c} histogram (clipped at 99.5%)")
                plt.xlabel(c)
                plt.ylabel("count")
                savefig(OUT_DIR / f"{c}_hist.png")

    # Receipt feature ↔ target correlations (train only)
    if "ed_cost_next3y_usd" in train2.columns:
        print_header("RECEIPT FEATURE ↔ TARGET (SPEARMAN)")
        y = train2["ed_cost_next3y_usd"].astype(float)
        receipt_cols = [c for c in train2.columns if c.startswith("receipt_") and pd.api.types.is_numeric_dtype(train2[c])]
        rows = [(c, spearman_corr(train2[c], y)) for c in receipt_cols]
        rc = pd.DataFrame(rows, columns=["receipt_feature", "spearman_r"]).sort_values("spearman_r", ascending=False)
        print(rc.head(20).to_string(index=False))
        rc.to_csv(OUT_DIR / "spearman_receipt_features.csv", index=False)

    # Code lift analysis (train only)
    if "ed_cost_next3y_usd" in train2.columns:
        print_header("CODE LIFT ANALYSIS (TRAIN ONLY)")
        pid_to_codes = {}
        for pid, obj in parsed.items():
            if isinstance(obj, dict):
                codes = [it.get("code", "") for it in obj.get("items", []) if isinstance(it, dict)]
            else:
                codes = []
            pid_to_codes[int(pid)] = set(codes)

        tr = train2.copy()
        if "prior_ed_cost_5y_usd" in tr.columns:
            tr["prior_cost_bin"] = pd.qcut(tr["prior_ed_cost_5y_usd"].astype(float), q=10, duplicates="drop")
        else:
            tr["prior_cost_bin"] = "all"

        group_keys = []
        if "primary_chronic" in tr.columns:
            group_keys.append("primary_chronic")
        group_keys.append("prior_cost_bin")

        baseline = tr.groupby(group_keys)["ed_cost_next3y_usd"].median().rename("baseline_median")
        tr = tr.join(baseline, on=group_keys)
        tr["residual_vs_baseline"] = tr["ed_cost_next3y_usd"].astype(float) - tr["baseline_median"].astype(float)

        overall_mean = tr["ed_cost_next3y_usd"].astype(float).mean()
        pid_to_y = dict(zip(tr["patient_id"].astype(int), tr["ed_cost_next3y_usd"].astype(float)))
        pid_to_resid = dict(zip(tr["patient_id"].astype(int), tr["residual_vs_baseline"].astype(float)))

        train_pids = set(tr["patient_id"].astype(int).tolist())
        code_patients = defaultdict(list)
        for pid in train_pids:
            for code in pid_to_codes.get(pid, set()):
                if code:
                    code_patients[code].append(pid)

        rows = []
        for code, pids in code_patients.items():
            n = len(pids)
            if n < CODE_MIN_SUPPORT:
                continue
            ys = np.array([pid_to_y[pid] for pid in pids], dtype=float)
            rs = np.array([pid_to_resid[pid] for pid in pids], dtype=float)
            rows.append({
                "code": code,
                "support_n": n,
                "mean_target": float(np.mean(ys)),
                "median_target": float(np.median(ys)),
                "lift_mean_vs_overall": float(np.mean(ys) - overall_mean),
                "mean_residual_vs_chronic_costbin_median": float(np.mean(rs)),
            })

        code_lift = pd.DataFrame(rows)
        if len(code_lift):
            code_lift = code_lift.sort_values(
                ["mean_residual_vs_chronic_costbin_median", "support_n"],
                ascending=[False, False]
            )
            code_lift.to_csv(OUT_DIR / "code_lift_table.csv", index=False)
            print("Top codes by residual lift (controls chronic + prior-cost-bin):")
            print(code_lift.head(N_TOP_CODES).to_string(index=False))
        else:
            print("No codes met support threshold. Consider lowering CODE_MIN_SUPPORT.")

    # Tail comparison
    if "ed_cost_next3y_usd" in train2.columns:
        print_header("TAIL COMPARISON (TOP 5% TARGET vs REST)")
        y = train2["ed_cost_next3y_usd"].astype(float)
        thr = y.quantile(0.95)
        top = train2[y >= thr].copy()
        rest = train2[y < thr].copy()
        print("Top5% threshold:", thr, "| top_n:", len(top), "| rest_n:", len(rest))

        key_feats = ["prior_ed_cost_5y_usd", "prior_ed_visits_5y",
                     "receipt_n_items", "receipt_n_unique_codes", "receipt_em_count",
                     "receipt_em_max", "receipt_top1_share", "receipt_bucket_entropy"]
        rows = []
        for f in key_feats:
            if f in train2.columns:
                a = pd.to_numeric(top[f], errors="coerce")
                b = pd.to_numeric(rest[f], errors="coerce")
                rows.append({
                    "feature": f,
                    "top_mean": float(a.mean()),
                    "rest_mean": float(b.mean()),
                    "delta_mean": float(a.mean() - b.mean()),
                    "top_p50": float(a.quantile(0.5)),
                    "rest_p50": float(b.quantile(0.5)),
                })
        tail_tbl = pd.DataFrame(rows).sort_values("delta_mean", ascending=False)
        print(tail_tbl.to_string(index=False))
        tail_tbl.to_csv(OUT_DIR / "tail_feature_comparison.csv", index=False)

    # Save merged tables
    train2.to_csv(OUT_DIR / "train_with_receipt_features.csv", index=False)
    test2.to_csv(OUT_DIR / "test_with_receipt_features.csv", index=False)

    print_header("DONE")
    print("Outputs saved to:", OUT_DIR)
    print("Cache used:", RECEIPT_CACHE)
    print("Key outputs:")
    print(" - pdf_total_mismatches_train/test.csv")
    print(" - spearman_receipt_features.csv")
    print(" - code_lift_table.csv")
    print(" - tail_feature_comparison.csv")

# Run
main()



LOAD DATA
Train shape: (2000, 5)
Test  shape: (2000, 4)
Train cols: ['patient_id', 'primary_chronic', 'prior_ed_visits_5y', 'prior_ed_cost_5y_usd', 'ed_cost_next3y_usd']
Patients shape: (4000, 5)
Patients cols: ['patient_id', 'age', 'sex', 'insurance', 'zip3']
Duplicate patient_id — train: 0 | test: 0
Train/Test patient_id overlap: 0

MISSINGNESS (TOP 30)
patient_id              0.0
primary_chronic         0.0
prior_ed_visits_5y      0.0
prior_ed_cost_5y_usd    0.0
ed_cost_next3y_usd      0.0
age                     0.0
sex                     0.0
insurance               0.0
zip3                    0.0

TARGET DISTRIBUTION
0.00      306.8800
0.01      991.4990
0.05     1553.6960
0.10     1867.5730
0.25     2548.7675
0.50     3569.0950
0.75     4956.4225
0.90     6636.4750
0.95     7541.8475
0.99     8801.0213
1.00    11184.6100
Mean: 3908.25191 Std: 1822.401920306473 Skew: 0.816103887108234
Top 5% contribute ~0.107 of total target spend

TARGET BY PRIMARY_CHRONIC
                 coun

Parsing PDFs: 100%|██████████| 4000/4000 [00:17<00:00, 224.83it/s]


Parsed 4000 receipts in 19.1s
Saved receipt cache: D:\AgentDs\agent_ds_healthcare\cache_iter10\receipts_parsed.joblib

RECEIPT FEATURE EXTRACTION
Receipt feature df: (4000, 15)

PDF TOTAL vs CSV prior_ed_cost_5y_usd VALIDATION
[train] match_rate=0.8950 | mismatches=210/2000
[test] match_rate=0.8900 | mismatches=220/2000

ZIP3 CONSISTENCY: patients.csv vs receipt PDFs (TRAIN)
ZIP3 match rate where both present: 1.0000 (n=2000)

INSURANCE CONSISTENCY: patients.csv vs receipt PDFs (TRAIN)
Insurance match rate where both present: 1.0000 (n=2000)

RECEIPT FEATURE SANITY (DISTRIBUTIONS)
receipt_n_items: mean=6.995 | p50=7.000 | p90=9.000 | missing=0.000
receipt_n_unique_codes: mean=5.122 | p50=5.000 | p90=7.000 | missing=0.000
receipt_em_count: mean=1.673 | p50=2.000 | p90=4.000 | missing=0.000
receipt_em_max: mean=3.531 | p50=4.000 | p90=5.000 | missing=0.280
receipt_top1_share: mean=0.311 | p50=0.290 | p90=0.471 | missing=0.000
receipt_bucket_entropy: mean=1.024 | p50=1.011 | p90=1.427 | m

## EDA Add-on

In [33]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
from joblib import load

BASE_DIR = Path(r"D:\AgentDs\agent_ds_healthcare")
TRAIN_CSV = BASE_DIR / "ed_cost_train.csv"
TEST_CSV  = BASE_DIR / "ed_cost_test.csv"
PATIENTS_CSV = BASE_DIR / "patients.csv"
CACHE_DIR = BASE_DIR / "cache_iter10"
RECEIPT_CACHE = CACHE_DIR / "receipts_parsed.joblib"

train = pd.read_csv(TRAIN_CSV)
test  = pd.read_csv(TEST_CSV)
patients = pd.read_csv(PATIENTS_CSV)

parsed = load(RECEIPT_CACHE)  # dict[patient_id] -> dict with "items", "pdf_total"

def sum_items(obj):
    if not isinstance(obj, dict):
        return np.nan
    items = obj.get("items", [])
    if not items:
        return 0.0
    return float(np.nansum([it.get("amount", np.nan) for it in items if isinstance(it, dict)]))

def code_counts(obj):
    if not isinstance(obj, dict):
        return {}
    items = obj.get("items", [])
    codes = [it.get("code","") for it in items if isinstance(it, dict)]
    cc = {}
    # ED E/M (99281-99285)
    em9928 = [c for c in codes if isinstance(c,str) and c.isdigit() and c.startswith("9928") and len(c)==5]
    cc["n_ed_em_9928x"] = len(em9928)
    cc["has_99285"] = int("99285" in em9928)
    # critical care
    cc["n_99291"] = sum(1 for c in codes if c=="99291")
    cc["n_99292"] = sum(1 for c in codes if c=="99292")
    cc["has_critical_care"] = int((cc["n_99291"] + cc["n_99292"]) > 0)
    # high acuity procedures
    cc["has_intub_31500"] = int("31500" in codes)
    cc["has_cpr_92950"] = int("92950" in codes)
    cc["has_cvc_36556"] = int("36556" in codes)
    cc["has_artline_36620"] = int("36620" in codes)
    # selected diagnostics from your lift list
    cc["has_ct_head_70450"] = int("70450" in codes)
    cc["has_ct_abdpel_74177"] = int("74177" in codes)
    cc["has_troponin_84484"] = int("84484" in codes)
    # observation
    cc["has_obs_G0378"] = int("G0378" in codes)
    return cc

# Build receipt table
rows = []
for pid, obj in parsed.items():
    pid = int(pid)
    r = {"patient_id": pid,
         "pdf_total": obj.get("pdf_total") if isinstance(obj, dict) else np.nan,
         "sum_items": sum_items(obj)}
    r.update(code_counts(obj))
    rows.append(r)
rec = pd.DataFrame(rows)

# Merge
tr = train.merge(patients, on="patient_id", how="left").merge(rec, on="patient_id", how="left")
te = test.merge(patients, on="patient_id", how="left").merge(rec, on="patient_id", how="left")

# --- A) TOTAL mismatch root cause ---
def match_rate(a, b, tol=0.01):
    d = (pd.to_numeric(a, errors="coerce") - pd.to_numeric(b, errors="coerce")).abs()
    return float((d <= tol).mean()), d

mr_total_tr, d_total_tr = match_rate(tr["pdf_total"], tr["prior_ed_cost_5y_usd"])
mr_sum_tr,   d_sum_tr   = match_rate(tr["sum_items"], tr["prior_ed_cost_5y_usd"])
print("TRAIN match_rate: pdf_total vs prior =", mr_total_tr, "| sum_items vs prior =", mr_sum_tr)

mr_total_te, d_total_te = match_rate(te["pdf_total"], te["prior_ed_cost_5y_usd"])
mr_sum_te,   d_sum_te   = match_rate(te["sum_items"], te["prior_ed_cost_5y_usd"])
print("TEST  match_rate: pdf_total vs prior =", mr_total_te, "| sum_items vs prior =", mr_sum_te)

# Show worst 15 mismatches (train) for pdf_total and for sum_items
tmp = tr[["patient_id","prior_ed_cost_5y_usd","pdf_total","sum_items"]].copy()
tmp["abs_diff_pdf_total"] = d_total_tr
tmp["abs_diff_sum_items"] = d_sum_tr
print("\nWorst 15 pdf_total mismatches (train):")
print(tmp.sort_values("abs_diff_pdf_total", ascending=False).head(15).to_string(index=False))

print("\nWorst 15 sum_items mismatches (train):")
print(tmp.sort_values("abs_diff_sum_items", ascending=False).head(15).to_string(index=False))

# --- B) Residual-lift check for new acuity features ---
# baseline residual: median by (primary_chronic x prior_cost_decile)
tr2 = tr.copy()
tr2["prior_cost_bin"] = pd.qcut(tr2["prior_ed_cost_5y_usd"], q=10, duplicates="drop")
base = tr2.groupby(["primary_chronic","prior_cost_bin"])["ed_cost_next3y_usd"].median().rename("baseline")
tr2 = tr2.join(base, on=["primary_chronic","prior_cost_bin"])
tr2["resid"] = tr2["ed_cost_next3y_usd"] - tr2["baseline"]

feat_list = [
    "has_critical_care","has_intub_31500","has_cpr_92950","has_cvc_36556","has_artline_36620",
    "has_ct_head_70450","has_ct_abdpel_74177","has_troponin_84484","has_obs_G0378","has_99285"
]

print("\nResidual lift vs baseline (mean resid), by feature=1:")
out = []
for f in feat_list:
    if f not in tr2.columns: 
        continue
    m = tr2[f]==1
    if m.sum() < 20:
        continue
    out.append((f, int(m.sum()), float(tr2.loc[m,"resid"].mean()), float(tr2.loc[m,"ed_cost_next3y_usd"].mean())))
out = pd.DataFrame(out, columns=["feature","support_n","mean_resid_lift","mean_target"]).sort_values("mean_resid_lift", ascending=False)
print(out.to_string(index=False))


TRAIN match_rate: pdf_total vs prior = 0.895 | sum_items vs prior = 1.0
TEST  match_rate: pdf_total vs prior = 0.89 | sum_items vs prior = 1.0

Worst 15 pdf_total mismatches (train):
 patient_id  prior_ed_cost_5y_usd  pdf_total  sum_items  abs_diff_pdf_total  abs_diff_sum_items
       1080               1031.30    1031.30    1031.30                 0.0        0.000000e+00
       1807               5664.15    5664.15    5664.15                 0.0        9.094947e-13
       3038               1025.28    1025.28    1025.28                 0.0        0.000000e+00
       1314               1852.77    1852.77    1852.77                 0.0        2.273737e-13
       3321               8489.80    8489.80    8489.80                 0.0        0.000000e+00
       1296               2912.06    2912.06    2912.06                 0.0        0.000000e+00
         33                735.11     735.11     735.11                 0.0        0.000000e+00
       3605               1737.56    1737.56    1

## Train

In [ ]:
# ============================================================
# AgentDS Challenge 2 — Robust XGBoost 3.0.0 Training (ONE CELL)
# Fixes:
#  - No pseudohuber (known to behave like null/constant in some setups)
#  - Early stopping moved to constructor (XGBoost sklearn breaking change)
#  - Stronger logs + baselines + sanity checks
# ============================================================

import os, re, time, json, math, warnings
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import scipy.sparse as sp

from joblib import load, dump
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction import FeatureHasher
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error

import xgboost as xgb

warnings.filterwarnings("ignore")
np.set_printoptions(suppress=True)

# -----------------------------
# Paths
# -----------------------------
BASE_DIR = Path(r"D:\AgentDs\agent_ds_healthcare")
TRAIN_CSV = BASE_DIR / "ed_cost_train.csv"
TEST_CSV  = BASE_DIR / "ed_cost_test.csv"
PATIENTS_CSV = BASE_DIR / "patients.csv"
CACHE_DIR = BASE_DIR / "cache_iter10"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RECEIPT_CACHE = CACHE_DIR / "receipts_parsed.joblib"
SUBMISSION_PATH = BASE_DIR / "submission_ICHI_V1.csv"

# -----------------------------
# Run + cache versions (prevents stale feature reuse)
# -----------------------------
PIPELINE_VERSION = "edcost_xgb_abs_es_ctor_v3_2026-02-14"
FEAT_CACHE = CACHE_DIR / "feat_matrix_cache_v3.joblib"  # new filename to avoid stale cache collisions

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = CACHE_DIR / f"run_{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

def log(msg):
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}] {msg}")

# -----------------------------
# Config
# -----------------------------
SEED = 42
N_SPLITS = 5

# sparse hashing of codes (captures long tail beyond curated codes)
USE_HASH_CODES = True
HASH_DIM = 2**16

# XGB training
N_ESTIMATORS = 5000
LEARNING_RATE = 0.05
MAX_DEPTH = 6
MIN_CHILD_WEIGHT = 10
SUBSAMPLE = 0.8
COLSAMPLE = 0.8
REG_LAMBDA = 1.0
REG_ALPHA = 0.0

EARLY_STOP = 250         # constructor-based
VERBOSE_EVAL = 200       # print every N rounds
WANT_CUDA = True         # will fall back to CPU automatically if GPU fails

# Calibration
USE_GROUP_MEDIAN_CALIBRATION = True
CALIB_BINS = 10

# -----------------------------
# Helpers
# -----------------------------
def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)

set_seed(SEED)

def normalize_cols(df):
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    return df

def safe_float(x):
    if x is None:
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    s = str(x).strip().replace("$","").replace(",","")
    try:
        return float(s)
    except Exception:
        return np.nan

def code_bucket(code: str) -> str:
    if not isinstance(code, str) or not code:
        return "unk"
    if code.isdigit():
        if code.startswith("99"):
            return "em_99"
        return f"num_{code[0]}"
    return f"alpha_{code[0].upper()}"

def entropy_from_counts(counts: Counter) -> float:
    if not counts:
        return 0.0
    vals = np.array(list(counts.values()), dtype=float)
    p = vals / (vals.sum() + 1e-12)
    return float(-(p * np.log(p + 1e-12)).sum())

def gini_from_amounts(x: np.ndarray) -> float:
    x = x.astype(float)
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return 0.0
    if np.all(x == 0):
        return 0.0
    x = np.sort(x)
    n = len(x)
    cumx = np.cumsum(x)
    g = (n + 1 - 2 * np.sum(cumx) / (cumx[-1] + 1e-12)) / n
    return float(g)

def build_prior_bins(prior: pd.Series, n_bins: int = 10):
    q = np.quantile(prior.values.astype(float), np.linspace(0, 1, n_bins + 1))
    bins = np.unique(q)
    if len(bins) < 3:
        bins = np.unique(np.quantile(prior.values.astype(float), [0, 0.5, 1.0]))
    return bins

def make_strat_labels(df: pd.DataFrame, n_bins: int = 10) -> np.ndarray:
    qbin = pd.qcut(df["prior_ed_cost_5y_usd"].astype(float), q=n_bins, duplicates="drop").astype(str)
    return (df["primary_chronic"].astype(str) + "|" + qbin).values

def predict_best(model, X):
    bi = getattr(model, "best_iteration", None)
    if bi is not None:
        try:
            return model.predict(X, iteration_range=(0, int(bi) + 1))
        except TypeError:
            pass
    return model.predict(X)

def sanity_stats(name, arr):
    arr = np.asarray(arr, dtype=float)
    return {
        f"{name}_min": float(np.nanmin(arr)),
        f"{name}_max": float(np.nanmax(arr)),
        f"{name}_mean": float(np.nanmean(arr)),
        f"{name}_p01": float(np.nanpercentile(arr, 1)),
        f"{name}_p50": float(np.nanpercentile(arr, 50)),
        f"{name}_p99": float(np.nanpercentile(arr, 99)),
    }

def ensure_finite(pred, context="pred"):
    if not np.all(np.isfinite(pred)):
        raise RuntimeError(f"[SANITY FAIL] Non-finite values in {context}.")
    if np.nanmax(np.abs(pred)) > 1e7:
        raise RuntimeError(f"[SANITY FAIL] {context} has extreme magnitude (>1e7). Likely objective/param failure.")

# -----------------------------
# Evidence-driven code lists
# -----------------------------
SEVERE_PROC = ["31500", "92950", "36556", "36620"]
CRIT_CARE   = ["99291", "99292"]
HI_DIAG     = ["70450", "74177", "84484"]
OBS         = ["G0378"]
ED_LEVELS   = ["99281", "99282", "99283", "99284", "99285"]
CURATED_CODES = SEVERE_PROC + CRIT_CARE + HI_DIAG + OBS + ED_LEVELS

# -----------------------------
# Receipt feature engineering (from receipts_parsed.joblib)
# -----------------------------
def receipt_features_from_items(obj: dict) -> dict:
    items = obj.get("items", []) if isinstance(obj, dict) else []
    if not items:
        out = {
            "n_items": 0, "n_unique_codes": 0,
            "top1_share": 0.0, "top3_share": 0.0, "distributed_cost": 0.0,
            "hhi": 1.0, "gini": 0.0, "bucket_entropy": 0.0,
            "n_em_9928x": 0, "max_em_level_9928x": 0.0,
            "has_critical_care": 0, "n_99291": 0, "n_99292": 0, "cc_time_proxy": 0.0,
            "n_severe_proc": 0, "severe_proc_share": 0.0,
            "n_hi_diag": 0, "hi_diag_share": 0.0,
            "severity_score": 0.0
        }
        for c in CURATED_CODES:
            out[f"has_{c}"] = 0
            out[f"cnt_{c}"] = 0
            out[f"share_{c}"] = 0.0
        return out

    codes = []
    amts = []
    for it in items:
        if not isinstance(it, dict):
            continue
        c = str(it.get("code","")).strip()
        a = safe_float(it.get("amount", np.nan))
        if c:
            codes.append(c)
        amts.append(a)
    amts = np.array(amts, dtype=float)
    total = float(np.nansum(amts))
    if not np.isfinite(total) or total <= 0:
        total = 1e-9

    cc = Counter(codes)
    n_items = int(len(items))
    n_unique = int(len(cc))

    top_sorted = np.sort(amts[~np.isnan(amts)])[::-1]
    top1 = float(top_sorted[0]) if len(top_sorted) else 0.0
    top3 = float(top_sorted[:3].sum()) if len(top_sorted) >= 3 else float(top_sorted.sum()) if len(top_sorted) else 0.0
    top1_share = top1 / total
    top3_share = top3 / total

    p = top_sorted / (top_sorted.sum() + 1e-12) if len(top_sorted) else np.array([1.0])
    hhi = float(np.sum(p * p)) if len(p) else 1.0
    gini = gini_from_amounts(amts)

    bucket_counts = Counter(code_bucket(c) for c in codes)
    bucket_entropy = entropy_from_counts(bucket_counts)

    em9928 = [c for c in codes if c in ED_LEVELS]
    n_em_9928x = len(em9928)
    max_em_level = float(max([int(c[-1]) for c in em9928], default=0))

    n_99291 = int(cc.get("99291", 0))
    n_99292 = int(cc.get("99292", 0))
    has_cc = int((n_99291 + n_99292) > 0)
    cc_time_proxy = float((74 if n_99291 > 0 else 0) + 30 * n_99292)

    n_severe = int(sum(cc.get(c, 0) for c in SEVERE_PROC))
    amt_by_code = Counter()
    for it in items:
        if not isinstance(it, dict):
            continue
        c = str(it.get("code","")).strip()
        a = safe_float(it.get("amount", 0.0))
        if c and np.isfinite(a):
            amt_by_code[c] += float(a)

    amt_severe = float(sum(amt_by_code.get(c, 0.0) for c in SEVERE_PROC))
    severe_share = amt_severe / total

    n_diag = int(sum(cc.get(c, 0) for c in HI_DIAG))
    amt_diag = float(sum(amt_by_code.get(c, 0.0) for c in HI_DIAG))
    diag_share = amt_diag / total

    has_intub = int(cc.get("31500", 0) > 0)
    has_cpr   = int(cc.get("92950", 0) > 0)
    has_cvc   = int(cc.get("36556", 0) > 0)
    has_art   = int(cc.get("36620", 0) > 0)
    has_cth   = int(cc.get("70450", 0) > 0)
    has_ctap  = int(cc.get("74177", 0) > 0)
    has_trop  = int(cc.get("84484", 0) > 0)

    severity_score = (
        5*has_intub + 5*has_cpr + 4*has_cvc + 4*has_art +
        3*has_cc + 1*has_cth + 1*has_ctap + 1*has_trop
    )

    out = {
        "n_items": n_items,
        "n_unique_codes": n_unique,
        "top1_share": float(top1_share),
        "top3_share": float(top3_share),
        "distributed_cost": float(1.0 - top3_share),
        "hhi": float(hhi),
        "gini": float(gini),
        "bucket_entropy": float(bucket_entropy),
        "n_em_9928x": int(n_em_9928x),
        "max_em_level_9928x": float(max_em_level),
        "has_critical_care": int(has_cc),
        "n_99291": int(n_99291),
        "n_99292": int(n_99292),
        "cc_time_proxy": float(cc_time_proxy),
        "n_severe_proc": int(n_severe),
        "severe_proc_share": float(severe_share),
        "n_hi_diag": int(n_diag),
        "hi_diag_share": float(diag_share),
        "severity_score": float(severity_score),
    }

    for c in CURATED_CODES:
        cnt = int(cc.get(c, 0))
        amt = float(amt_by_code.get(c, 0.0))
        out[f"has_{c}"] = int(cnt > 0)
        out[f"cnt_{c}"] = cnt
        out[f"share_{c}"] = float(amt / total)

    return out

def build_code_hash_dict(obj: dict) -> dict:
    items = obj.get("items", []) if isinstance(obj, dict) else []
    codes = []
    for it in items:
        if isinstance(it, dict):
            c = str(it.get("code","")).strip()
            if c:
                codes.append(c)
    cc = Counter(codes)
    bc = Counter(code_bucket(c) for c in codes)

    d = {}
    for c, v in cc.items():
        d[f"c={c}"] = float(v)
    for b, v in bc.items():
        d[f"b={b}"] = float(v)
    return d

# -----------------------------
# Build feature matrices
# -----------------------------
def build_features(train_df, test_df, patients_df, receipts_dict):
    train_df = train_df.merge(patients_df, on="patient_id", how="left")
    test_df  = test_df.merge(patients_df, on="patient_id", how="left")

    all_ids = pd.concat([train_df["patient_id"], test_df["patient_id"]], axis=0).astype(int).tolist()

    rec_rows = []
    hash_dicts = []
    missing = 0
    for pid in all_ids:
        obj = receipts_dict.get(int(pid))
        if obj is None:
            missing += 1
            obj = {"items": []}
        rec_rows.append({"patient_id": int(pid), **receipt_features_from_items(obj)})
        if USE_HASH_CODES:
            hash_dicts.append(build_code_hash_dict(obj))

    if missing:
        log(f"[WARN] Missing receipts for {missing} ids; filled with empty receipts.")

    rec_feat = pd.DataFrame(rec_rows)
    rec_train = rec_feat.iloc[:len(train_df)].copy()
    rec_test  = rec_feat.iloc[len(train_df):].copy()

    train_df = train_df.merge(rec_train, on="patient_id", how="left")
    test_df  = test_df.merge(rec_test, on="patient_id", how="left")

    # Tabular engineered features
    train_df["prior_cost_per_year"]   = train_df["prior_ed_cost_5y_usd"] / 5.0
    test_df["prior_cost_per_year"]    = test_df["prior_ed_cost_5y_usd"] / 5.0
    train_df["prior_visits_per_year"] = train_df["prior_ed_visits_5y"] / 5.0
    test_df["prior_visits_per_year"]  = test_df["prior_ed_visits_5y"] / 5.0
    train_df["prior_cost_per_visit"]  = train_df["prior_ed_cost_5y_usd"] / (train_df["prior_ed_visits_5y"] + 1e-6)
    test_df["prior_cost_per_visit"]   = test_df["prior_ed_cost_5y_usd"] / (test_df["prior_ed_visits_5y"] + 1e-6)

    cat_cols = ["primary_chronic", "sex", "insurance", "zip3"]
    for c in cat_cols:
        train_df[c] = train_df[c].astype(str)
        test_df[c]  = test_df[c].astype(str)

    base_num = [
        "age",
        "prior_ed_visits_5y",
        "prior_ed_cost_5y_usd",
        "prior_cost_per_year",
        "prior_visits_per_year",
        "prior_cost_per_visit",
    ]
    receipt_num = [
        "n_items","n_unique_codes","top1_share","top3_share","distributed_cost",
        "hhi","gini","bucket_entropy",
        "n_em_9928x","max_em_level_9928x",
        "has_critical_care","n_99291","n_99292","cc_time_proxy",
        "n_severe_proc","severe_proc_share",
        "n_hi_diag","hi_diag_share",
        "severity_score",
    ]
    curated = []
    for c in CURATED_CODES:
        curated += [f"has_{c}", f"cnt_{c}", f"share_{c}"]

    num_cols = base_num + receipt_num + curated

    for df in [train_df, test_df]:
        for c in num_cols:
            if c not in df.columns:
                df[c] = 0.0
        df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).astype(np.float32)

    all_cat = pd.concat([train_df[cat_cols], test_df[cat_cols]], axis=0, ignore_index=True)

    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=True)

    X_cat_all = ohe.fit_transform(all_cat)
    X_num_all = sp.csr_matrix(pd.concat([train_df[num_cols], test_df[num_cols]], axis=0, ignore_index=True).values)

    if USE_HASH_CODES:
        hasher = FeatureHasher(n_features=HASH_DIM, input_type="dict")
        X_hash_all = hasher.transform(hash_dicts)
        X_all = sp.hstack([X_num_all, X_cat_all, X_hash_all], format="csr")
    else:
        X_all = sp.hstack([X_num_all, X_cat_all], format="csr")

    X_train = X_all[:len(train_df)]
    X_test  = X_all[len(train_df):]
    return train_df, test_df, X_train, X_test

# -----------------------------
# XGB constructor params (XGBoost 3 requires early_stopping_rounds in ctor)
# -----------------------------
def make_xgb_regressor(seed: int, use_cuda: bool = True):
    params = dict(
        n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE,
        max_depth=MAX_DEPTH,
        min_child_weight=MIN_CHILD_WEIGHT,
        subsample=SUBSAMPLE,
        colsample_bytree=COLSAMPLE,
        reg_lambda=REG_LAMBDA,
        reg_alpha=REG_ALPHA,
        random_state=seed,
        n_jobs=-1,
        verbosity=1,
        objective="reg:absoluteerror",
        eval_metric="mae",
        early_stopping_rounds=EARLY_STOP,   # <-- KEY FIX for XGBoost >=2.1.4 / 3.x
        tree_method="hist",
    )
    if use_cuda:
        params["device"] = "cuda"  # GPU device param
    else:
        params["device"] = "cpu"
    return xgb.XGBRegressor(**params)

def group_median_baseline(train_df, y_tr, idx_tr, idx_va, bins):
    tr = train_df.iloc[idx_tr].copy()
    va = train_df.iloc[idx_va].copy()
    tr_bin = pd.cut(tr["prior_ed_cost_5y_usd"].astype(float), bins=bins, include_lowest=True).astype(str)
    va_bin = pd.cut(va["prior_ed_cost_5y_usd"].astype(float), bins=bins, include_lowest=True).astype(str)
    tr_key = (tr["primary_chronic"].astype(str) + "|" + tr_bin).values
    va_key = (va["primary_chronic"].astype(str) + "|" + va_bin).values

    med = pd.Series(y_tr).groupby(tr_key).median()
    med_dict = med.to_dict()
    global_med = float(np.median(y_tr))
    pred_va = np.array([med_dict.get(k, global_med) for k in va_key], dtype=float)
    return pred_va

def group_median_calibrate(train_df, oof_pred, y_true, test_df, test_pred, n_bins: int = 10):
    bins = build_prior_bins(train_df["prior_ed_cost_5y_usd"].astype(float), n_bins=n_bins)
    tr_bin = pd.cut(train_df["prior_ed_cost_5y_usd"].astype(float), bins=bins, include_lowest=True).astype(str)
    te_bin = pd.cut(test_df["prior_ed_cost_5y_usd"].astype(float), bins=bins, include_lowest=True).astype(str)
    tr_key = (train_df["primary_chronic"].astype(str) + "|" + tr_bin).values
    te_key = (test_df["primary_chronic"].astype(str) + "|" + te_bin).values

    resid = y_true - oof_pred
    shift = pd.Series(resid).groupby(tr_key).median().to_dict()

    oof_adj = oof_pred + np.array([shift.get(k, 0.0) for k in tr_key], dtype=float)
    te_adj  = test_pred + np.array([shift.get(k, 0.0) for k in te_key], dtype=float)
    return oof_adj, te_adj, shift

# ============================================================
# MAIN
# ============================================================
log(f"xgboost version: {getattr(xgb, '__version__', 'unknown')}")
log(f"PIPELINE_VERSION: {PIPELINE_VERSION}")
log("Loading CSVs...")
train = normalize_cols(pd.read_csv(TRAIN_CSV))
test  = normalize_cols(pd.read_csv(TEST_CSV))
patients = normalize_cols(pd.read_csv(PATIENTS_CSV))

assert RECEIPT_CACHE.exists(), f"Missing {RECEIPT_CACHE}"
log("Loading receipts cache...")
receipts = load(RECEIPT_CACHE)
if not isinstance(receipts, dict) or len(receipts) < 3000:
    raise RuntimeError("receipts_parsed.joblib has unexpected structure/size. Rebuild it first.")

# Load/build feature matrices
need_rebuild = True
if FEAT_CACHE.exists():
    try:
        cache = load(FEAT_CACHE)
        if cache.get("version") == PIPELINE_VERSION:
            train2 = cache["train2"]; test2 = cache["test2"]
            X_train = cache["X_train"]; X_test = cache["X_test"]
            need_rebuild = False
            log(f"Loaded feature cache: {FEAT_CACHE}")
    except Exception:
        need_rebuild = True

if need_rebuild:
    log("Building features (cache miss/version change)...")
    train2, test2, X_train, X_test = build_features(train, test, patients, receipts)
    dump({"version": PIPELINE_VERSION, "train2": train2, "test2": test2, "X_train": X_train, "X_test": X_test}, FEAT_CACHE)
    log(f"Saved feature cache: {FEAT_CACHE}")

y = train2["ed_cost_next3y_usd"].astype(float).values
log(f"X_train shape={X_train.shape}, nnz={X_train.nnz}, density={X_train.nnz/(X_train.shape[0]*X_train.shape[1]):.6f}")
log(f"Target stats: mean={y.mean():.3f}, p50={np.percentile(y,50):.3f}, p95={np.percentile(y,95):.3f}, max={y.max():.3f}")

# CV
strat = make_strat_labels(train2, n_bins=10)
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof = np.zeros(len(train2), dtype=float)
test_pred = np.zeros(len(test2), dtype=float)

global_bins = build_prior_bins(train2["prior_ed_cost_5y_usd"].astype(float), n_bins=10)

fold_logs = []
log("Starting CV training...")

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(train2)), strat), 1):
    t_fold = time.time()
    X_tr, X_va = X_train[tr_idx], X_train[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    # Baseline: group-median by (primary_chronic x prior_cost_bin)
    base_va = group_median_baseline(train2, y_tr, tr_idx, va_idx, bins=global_bins)
    base_mae = mean_absolute_error(y_va, base_va)

    # Train model
    model = make_xgb_regressor(seed=SEED + fold, use_cuda=WANT_CUDA)
    try:
        # eval_set includes train + valid for better logs (valid is validation_1)
        model.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_va, y_va)], verbose=VERBOSE_EVAL)
        train_mode = "cuda" if WANT_CUDA else "cpu"
    except xgb.core.XGBoostError as e:
        log(f"[Fold {fold}] GPU training failed; falling back to CPU. Error: {repr(e)[:200]}")
        model = make_xgb_regressor(seed=SEED + fold, use_cuda=False)
        model.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_va, y_va)], verbose=VERBOSE_EVAL)
        train_mode = "cpu"

    pred_va = predict_best(model, X_va)
    pred_te = predict_best(model, X_test)

    # Sanity checks
    ensure_finite(pred_va, context=f"fold{fold}_pred_va")
    ensure_finite(pred_te, context=f"fold{fold}_pred_te")

    # Clip to non-negative costs
    pred_va = np.clip(pred_va, 0.0, None)
    pred_te = np.clip(pred_te, 0.0, None)

    fold_mae = mean_absolute_error(y_va, pred_va)
    oof[va_idx] = pred_va
    test_pred += pred_te / N_SPLITS

    best_iter = getattr(model, "best_iteration", None)
    best_score = getattr(model, "best_score", None)

    info = {
        "fold": fold,
        "train_mode": train_mode,
        "n_tr": int(len(tr_idx)),
        "n_va": int(len(va_idx)),
        "baseline_group_median_mae": float(base_mae),
        "model_mae": float(fold_mae),
        "mae_improvement_vs_baseline": float(base_mae - fold_mae),
        "best_iteration": None if best_iter is None else int(best_iter),
        "best_score": None if best_score is None else float(best_score),
        "time_sec": float(time.time() - t_fold),
        **sanity_stats("pred_va", pred_va),
        **sanity_stats("y_va", y_va),
    }
    fold_logs.append(info)

    log(f"[Fold {fold}/{N_SPLITS}] mode={train_mode} | baseline_mae={base_mae:.3f} | model_mae={fold_mae:.3f} | "
        f"improve={base_mae-fold_mae:.3f} | best_iter={info['best_iteration']} | time={info['time_sec']:.1f}s")

oof_mae = mean_absolute_error(y, oof)
log(f"OOF MAE (raw): {oof_mae:.6f}")

# Optional calibration
if USE_GROUP_MEDIAN_CALIBRATION:
    log("Applying group-median residual calibration (primary_chronic x prior_cost_bin)...")
    oof_cal, test_cal, shift = group_median_calibrate(train2, oof, y, test2, test_pred, n_bins=CALIB_BINS)
    oof_mae_cal = mean_absolute_error(y, oof_cal)
    log(f"OOF MAE calibrated: {oof_mae_cal:.6f} (delta={oof_mae - oof_mae_cal:+.6f})")
    test_pred_final = np.clip(test_cal, 0.0, None)
else:
    test_pred_final = np.clip(test_pred, 0.0, None)

# Save outputs
oof_df = pd.DataFrame({"patient_id": train2["patient_id"].astype(int), "y_true": y, "oof_pred": oof})
oof_df.to_csv(RUN_DIR / "oof_predictions.csv", index=False)

sub = pd.DataFrame({"patient_id": test2["patient_id"].astype(int), "ed_cost_next3y_usd": test_pred_final})
sub.to_csv(RUN_DIR / "submission.csv", index=False)
sub.to_csv(SUBMISSION_PATH, index=False)

run_log = {
    "run_id": RUN_ID,
    "pipeline_version": PIPELINE_VERSION,
    "xgboost_version": getattr(xgb, "__version__", "unknown"),
    "use_cuda_requested": WANT_CUDA,
    "n_splits": N_SPLITS,
    "feature_matrix_shape": [int(X_train.shape[0]), int(X_train.shape[1])],
    "feature_matrix_nnz": int(X_train.nnz),
    "oof_mae_raw": float(oof_mae),
    "oof_mae_calibrated": float(oof_mae_cal) if USE_GROUP_MEDIAN_CALIBRATION else None,
    "folds": fold_logs,
}
with open(RUN_DIR / "run_log.json", "w", encoding="utf-8") as f:
    json.dump(run_log, f, indent=2)

log(f"Saved run artifacts to: {RUN_DIR}")
log(f"Saved submission -> {SUBMISSION_PATH}")


[06:21:11] xgboost version: 3.0.0
[06:21:11] PIPELINE_VERSION: edcost_xgb_abs_es_ctor_v3_2026-02-14
[06:21:11] Loading CSVs...
[06:21:11] Loading receipts cache...
[06:21:12] Building features (cache miss/version change)...
[06:21:12] Saved feature cache: D:\AgentDs\agent_ds_healthcare\cache_iter10\feat_matrix_cache_v3.joblib
[06:21:12] X_train shape=(2000, 65623), nnz=90215, density=0.000687
[06:21:12] Target stats: mean=3908.252, p50=3569.095, p95=7541.847, max=11184.610
[06:21:12] Starting CV training...
[0]	validation_0-mae:1374.61808	validation_1-mae:1363.71082
[200]	validation_0-mae:241.06473	validation_1-mae:459.63806
[390]	validation_0-mae:191.44000	validation_1-mae:460.98083
[06:21:38] [Fold 1/5] mode=cuda | baseline_mae=510.842 | model_mae=458.891 | improve=51.952 | best_iter=140 | time=25.9s
[0]	validation_0-mae:1365.86338	validation_1-mae:1398.37807
[200]	validation_0-mae:244.07556	validation_1-mae:442.20840
[400]	validation_0-mae:192.75596	validation_1-mae:439.24362
[600]	

# Iteration 2
- 462.0000

In [25]:
# ============================================================
# AgentDS Challenge 2 — V2: Shift-Aware Training (Adversarial Importance Weighting)
# One-cell pipeline, XGBoost 3.0.0 compatible, robust logging.
#
# Core change vs V1:
#   - Adversarial validation (train vs test) -> importance weights
#   - Train regressor with sample_weight to match test distribution
#
# Theory refs:
#   - Covariate shift / importance weighting: Sugiyama (JMLR 2007)
#   - Adversarial validation practice: Kaggle notebooks
# ============================================================

import os, re, time, json, math, warnings, inspect
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import scipy.sparse as sp

from joblib import load, dump
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction import FeatureHasher
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error, roc_auc_score

import xgboost as xgb

warnings.filterwarnings("ignore")
np.set_printoptions(suppress=True)

# -----------------------------
# Paths
# -----------------------------
BASE_DIR = Path(r"D:\AgentDs\agent_ds_healthcare")
TRAIN_CSV = BASE_DIR / "ed_cost_train.csv"
TEST_CSV  = BASE_DIR / "ed_cost_test.csv"
PATIENTS_CSV = BASE_DIR / "patients.csv"
CACHE_DIR = BASE_DIR / "cache_iter10"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RECEIPT_CACHE = CACHE_DIR / "receipts_parsed.joblib"
SUBMISSION_PATH = BASE_DIR / "submission_ICHI_V1.csv"

# New caches
PIPELINE_VERSION = "EDCOST_V2_adv_weighting_2026-02-14"
FEAT_CACHE = CACHE_DIR / "feat_matrix_cache_v2.joblib"
ADV_CACHE  = CACHE_DIR / "adv_scores_cache_v2.joblib"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = CACHE_DIR / f"run_v2_{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

def log(msg):
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}] {msg}")

# -----------------------------
# Config
# -----------------------------
SEED = 42
N_SPLITS_REG = 5
N_SPLITS_ADV = 5

# Features
USE_HASH_CODES = True
HASH_DIM = 2**16

# Importance weights (clip to avoid instability)
W_EPS = 1e-3
W_CLIP_MIN = 0.25
W_CLIP_MAX = 4.0

# XGB (regression)
REG_PARAMS = dict(
    n_estimators=6000,
    learning_rate=0.04,
    max_depth=6,
    min_child_weight=20,      # a bit more conservative than V1
    subsample=0.8,
    colsample_bytree=0.75,
    reg_lambda=5.0,
    reg_alpha=0.1,
    objective="reg:absoluteerror",
    eval_metric="mae",
    early_stopping_rounds=300,
    tree_method="hist",
    device="cuda",            # will fall back to cpu if needed
    n_jobs=-1,
    verbosity=1,
)

# XGB (adversarial classifier)
ADV_PARAMS = dict(
    n_estimators=2500,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5.0,
    reg_alpha=0.0,
    objective="binary:logistic",
    eval_metric="auc",
    early_stopping_rounds=150,
    tree_method="hist",
    device="cuda",
    n_jobs=-1,
    verbosity=0,
)

VERBOSE_EVAL = 200

# Calibration
USE_WEIGHTED_GROUP_CALIBRATION = True
CALIB_BINS = 10

# -----------------------------
# Helpers
# -----------------------------
def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
set_seed(SEED)

def normalize_cols(df):
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    return df

def safe_float(x):
    if x is None:
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    s = str(x).strip().replace("$","").replace(",","")
    try:
        return float(s)
    except Exception:
        return np.nan

def code_bucket(code: str) -> str:
    if not isinstance(code, str) or not code:
        return "unk"
    if code.isdigit():
        if code.startswith("99"):
            return "em_99"
        return f"num_{code[0]}"
    return f"alpha_{code[0].upper()}"

def entropy_from_counts(counts: Counter) -> float:
    if not counts:
        return 0.0
    vals = np.array(list(counts.values()), dtype=float)
    p = vals / (vals.sum() + 1e-12)
    return float(-(p * np.log(p + 1e-12)).sum())

def gini_from_amounts(x: np.ndarray) -> float:
    x = x.astype(float)
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return 0.0
    if np.all(x == 0):
        return 0.0
    x = np.sort(x)
    n = len(x)
    cumx = np.cumsum(x)
    g = (n + 1 - 2 * np.sum(cumx) / (cumx[-1] + 1e-12)) / n
    return float(g)

def build_prior_bins(prior: pd.Series, n_bins: int = 10):
    q = np.quantile(prior.values.astype(float), np.linspace(0, 1, n_bins + 1))
    bins = np.unique(q)
    if len(bins) < 3:
        bins = np.unique(np.quantile(prior.values.astype(float), [0, 0.5, 1.0]))
    return bins

def weighted_mae(y_true, y_pred, w):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    w = np.asarray(w, dtype=float)
    w = np.clip(w, 1e-12, None)
    return float(np.sum(w * np.abs(y_true - y_pred)) / np.sum(w))

def weighted_median(values, weights):
    v = np.asarray(values, dtype=float)
    w = np.asarray(weights, dtype=float)
    m = np.isfinite(v) & np.isfinite(w) & (w > 0)
    v, w = v[m], w[m]
    if len(v) == 0:
        return 0.0
    idx = np.argsort(v)
    v, w = v[idx], w[idx]
    cdf = np.cumsum(w)
    cutoff = 0.5 * np.sum(w)
    j = int(np.searchsorted(cdf, cutoff, side="left"))
    j = min(max(j, 0), len(v)-1)
    return float(v[j])

def predict_best(model, X):
    bi = getattr(model, "best_iteration", None)
    if bi is not None:
        try:
            return model.predict(X, iteration_range=(0, int(bi) + 1))
        except TypeError:
            pass
    return model.predict(X)

def predict_best_proba(model, X):
    bi = getattr(model, "best_iteration", None)
    if bi is not None:
        try:
            return model.predict_proba(X, iteration_range=(0, int(bi) + 1))[:, 1]
        except TypeError:
            pass
    return model.predict_proba(X)[:, 1]

def safe_fit(model, X_tr, y_tr, eval_set=None, sample_weight=None, sample_weight_eval_set=None, verbose=200):
    sig = inspect.signature(model.fit)
    kwargs = {}
    if "eval_set" in sig.parameters and eval_set is not None:
        kwargs["eval_set"] = eval_set
    if "sample_weight" in sig.parameters and sample_weight is not None:
        kwargs["sample_weight"] = sample_weight
    if "sample_weight_eval_set" in sig.parameters and sample_weight_eval_set is not None:
        kwargs["sample_weight_eval_set"] = sample_weight_eval_set
    if "verbose" in sig.parameters:
        kwargs["verbose"] = verbose
    model.fit(X_tr, y_tr, **kwargs)
    return model

def sanity_stats(arr):
    arr = np.asarray(arr, dtype=float)
    return dict(
        min=float(np.min(arr)),
        max=float(np.max(arr)),
        mean=float(np.mean(arr)),
        p01=float(np.percentile(arr, 1)),
        p50=float(np.percentile(arr, 50)),
        p99=float(np.percentile(arr, 99)),
    )

# -----------------------------
# Evidence-driven code lists
# -----------------------------
SEVERE_PROC = ["31500", "92950", "36556", "36620"]
CRIT_CARE   = ["99291", "99292"]
HI_DIAG     = ["70450", "74177", "84484"]
OBS         = ["G0378"]
ED_LEVELS   = ["99281", "99282", "99283", "99284", "99285"]
CURATED_CODES = SEVERE_PROC + CRIT_CARE + HI_DIAG + OBS + ED_LEVELS

# -----------------------------
# Receipt features from receipts_parsed.joblib
# -----------------------------
def receipt_features_from_items(obj: dict) -> dict:
    items = obj.get("items", []) if isinstance(obj, dict) else []
    if not items:
        out = {
            "n_items": 0, "n_unique_codes": 0,
            "top1_share": 0.0, "top3_share": 0.0, "distributed_cost": 0.0,
            "hhi": 1.0, "gini": 0.0, "bucket_entropy": 0.0,
            "n_em_9928x": 0, "max_em_level_9928x": 0.0,
            "has_critical_care": 0, "n_99291": 0, "n_99292": 0, "cc_time_proxy": 0.0,
            "n_severe_proc": 0, "severe_proc_share": 0.0,
            "n_hi_diag": 0, "hi_diag_share": 0.0,
            "severity_score": 0.0
        }
        for c in CURATED_CODES:
            out[f"has_{c}"] = 0
            out[f"cnt_{c}"] = 0
            out[f"share_{c}"] = 0.0
        return out

    codes, amts = [], []
    for it in items:
        if not isinstance(it, dict):
            continue
        c = str(it.get("code","")).strip()
        a = safe_float(it.get("amount", np.nan))
        if c:
            codes.append(c)
        amts.append(a)
    amts = np.array(amts, dtype=float)
    total = float(np.nansum(amts))
    if not np.isfinite(total) or total <= 0:
        total = 1e-9

    cc = Counter(codes)
    n_items = int(len(items))
    n_unique = int(len(cc))

    top_sorted = np.sort(amts[~np.isnan(amts)])[::-1]
    top1 = float(top_sorted[0]) if len(top_sorted) else 0.0
    top3 = float(top_sorted[:3].sum()) if len(top_sorted) >= 3 else float(top_sorted.sum()) if len(top_sorted) else 0.0
    top1_share = top1 / total
    top3_share = top3 / total

    p = top_sorted / (top_sorted.sum() + 1e-12) if len(top_sorted) else np.array([1.0])
    hhi = float(np.sum(p * p)) if len(p) else 1.0
    gini = gini_from_amounts(amts)

    bucket_counts = Counter(code_bucket(c) for c in codes)
    bucket_entropy = entropy_from_counts(bucket_counts)

    em9928 = [c for c in codes if c in ED_LEVELS]
    n_em_9928x = len(em9928)
    max_em_level = float(max([int(c[-1]) for c in em9928], default=0))

    n_99291 = int(cc.get("99291", 0))
    n_99292 = int(cc.get("99292", 0))
    has_cc  = int((n_99291 + n_99292) > 0)
    cc_time_proxy = float((74 if n_99291 > 0 else 0) + 30 * n_99292)

    n_severe = int(sum(cc.get(c, 0) for c in SEVERE_PROC))

    amt_by_code = Counter()
    for it in items:
        if not isinstance(it, dict):
            continue
        c = str(it.get("code","")).strip()
        a = safe_float(it.get("amount", 0.0))
        if c and np.isfinite(a):
            amt_by_code[c] += float(a)

    amt_severe = float(sum(amt_by_code.get(c, 0.0) for c in SEVERE_PROC))
    severe_share = amt_severe / total

    n_diag = int(sum(cc.get(c, 0) for c in HI_DIAG))
    amt_diag = float(sum(amt_by_code.get(c, 0.0) for c in HI_DIAG))
    diag_share = amt_diag / total

    has_intub = int(cc.get("31500", 0) > 0)
    has_cpr   = int(cc.get("92950", 0) > 0)
    has_cvc   = int(cc.get("36556", 0) > 0)
    has_art   = int(cc.get("36620", 0) > 0)
    has_cth   = int(cc.get("70450", 0) > 0)
    has_ctap  = int(cc.get("74177", 0) > 0)
    has_trop  = int(cc.get("84484", 0) > 0)

    severity_score = (
        5*has_intub + 5*has_cpr + 4*has_cvc + 4*has_art +
        3*has_cc + 1*has_cth + 1*has_ctap + 1*has_trop
    )

    out = {
        "n_items": n_items,
        "n_unique_codes": n_unique,
        "top1_share": float(top1_share),
        "top3_share": float(top3_share),
        "distributed_cost": float(1.0 - top3_share),
        "hhi": float(hhi),
        "gini": float(gini),
        "bucket_entropy": float(bucket_entropy),
        "n_em_9928x": int(n_em_9928x),
        "max_em_level_9928x": float(max_em_level),
        "has_critical_care": int(has_cc),
        "n_99291": int(n_99291),
        "n_99292": int(n_99292),
        "cc_time_proxy": float(cc_time_proxy),
        "n_severe_proc": int(n_severe),
        "severe_proc_share": float(severe_share),
        "n_hi_diag": int(n_diag),
        "hi_diag_share": float(diag_share),
        "severity_score": float(severity_score),
    }

    for c in CURATED_CODES:
        cnt = int(cc.get(c, 0))
        amt = float(amt_by_code.get(c, 0.0))
        out[f"has_{c}"] = int(cnt > 0)
        out[f"cnt_{c}"] = cnt
        out[f"share_{c}"] = float(amt / total)

    return out

def build_code_hash_dict(obj: dict) -> dict:
    items = obj.get("items", []) if isinstance(obj, dict) else []
    codes = []
    for it in items:
        if isinstance(it, dict):
            c = str(it.get("code","")).strip()
            if c:
                codes.append(c)
    cc = Counter(codes)
    bc = Counter(code_bucket(c) for c in codes)
    d = {}
    for c, v in cc.items():
        d[f"c={c}"] = float(v)
    for b, v in bc.items():
        d[f"b={b}"] = float(v)
    return d

# -----------------------------
# Feature build
# -----------------------------
def build_features(train_df, test_df, patients_df, receipts_dict):
    train_df = train_df.merge(patients_df, on="patient_id", how="left")
    test_df  = test_df.merge(patients_df, on="patient_id", how="left")

    all_ids = pd.concat([train_df["patient_id"], test_df["patient_id"]], axis=0).astype(int).tolist()

    rec_rows, hash_dicts = [], []
    for pid in all_ids:
        obj = receipts_dict.get(int(pid), {"items": []})
        rec_rows.append({"patient_id": int(pid), **receipt_features_from_items(obj)})
        if USE_HASH_CODES:
            hash_dicts.append(build_code_hash_dict(obj))

    rec_feat = pd.DataFrame(rec_rows)
    rec_train = rec_feat.iloc[:len(train_df)].copy()
    rec_test  = rec_feat.iloc[len(train_df):].copy()

    train_df = train_df.merge(rec_train, on="patient_id", how="left")
    test_df  = test_df.merge(rec_test, on="patient_id", how="left")

    # tabular engineered
    train_df["prior_cost_per_year"]   = train_df["prior_ed_cost_5y_usd"] / 5.0
    test_df["prior_cost_per_year"]    = test_df["prior_ed_cost_5y_usd"] / 5.0
    train_df["prior_visits_per_year"] = train_df["prior_ed_visits_5y"] / 5.0
    test_df["prior_visits_per_year"]  = test_df["prior_ed_visits_5y"] / 5.0
    train_df["prior_cost_per_visit"]  = train_df["prior_ed_cost_5y_usd"] / (train_df["prior_ed_visits_5y"] + 1e-6)
    test_df["prior_cost_per_visit"]   = test_df["prior_ed_cost_5y_usd"] / (test_df["prior_ed_visits_5y"] + 1e-6)

    # categoricals
    cat_cols = ["primary_chronic", "sex", "insurance", "zip3"]
    for c in cat_cols:
        train_df[c] = train_df[c].astype(str)
        test_df[c]  = test_df[c].astype(str)

    base_num = [
        "age",
        "prior_ed_visits_5y",
        "prior_ed_cost_5y_usd",
        "prior_cost_per_year",
        "prior_visits_per_year",
        "prior_cost_per_visit",
    ]
    receipt_num = [
        "n_items","n_unique_codes","top1_share","top3_share","distributed_cost",
        "hhi","gini","bucket_entropy",
        "n_em_9928x","max_em_level_9928x",
        "has_critical_care","n_99291","n_99292","cc_time_proxy",
        "n_severe_proc","severe_proc_share",
        "n_hi_diag","hi_diag_share",
        "severity_score",
    ]
    curated = []
    for c in CURATED_CODES:
        curated += [f"has_{c}", f"cnt_{c}", f"share_{c}"]

    num_cols = base_num + receipt_num + curated

    for df in [train_df, test_df]:
        for c in num_cols:
            if c not in df.columns:
                df[c] = 0.0
        df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).astype(np.float32)

    all_cat = pd.concat([train_df[cat_cols], test_df[cat_cols]], axis=0, ignore_index=True)
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=True)

    X_cat_all = ohe.fit_transform(all_cat)
    X_num_all = sp.csr_matrix(pd.concat([train_df[num_cols], test_df[num_cols]], axis=0, ignore_index=True).values)

    if USE_HASH_CODES:
        hasher = FeatureHasher(n_features=HASH_DIM, input_type="dict")
        X_hash_all = hasher.transform(hash_dicts)
        X_all = sp.hstack([X_num_all, X_cat_all, X_hash_all], format="csr")
    else:
        X_all = sp.hstack([X_num_all, X_cat_all], format="csr")

    X_train = X_all[:len(train_df)]
    X_test  = X_all[len(train_df):]
    return train_df, test_df, X_train, X_test

# -----------------------------
# Adversarial validation -> weights
# -----------------------------
def adversarial_oof_scores(X_train, X_test):
    X_all = sp.vstack([X_train, X_test], format="csr")
    y_adv = np.concatenate([np.zeros(X_train.shape[0], dtype=int), np.ones(X_test.shape[0], dtype=int)])

    skf = StratifiedKFold(n_splits=N_SPLITS_ADV, shuffle=True, random_state=SEED)
    oof = np.zeros(len(y_adv), dtype=float)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y_adv)), y_adv), 1):
        clf = xgb.XGBClassifier(**{**ADV_PARAMS, "random_state": SEED + fold})
        try:
            clf = safe_fit(clf, X_all[tr_idx], y_adv[tr_idx],
                           eval_set=[(X_all[va_idx], y_adv[va_idx])],
                           verbose=False)
        except xgb.core.XGBoostError:
            # CPU fallback
            p = dict(ADV_PARAMS); p["device"] = "cpu"
            clf = xgb.XGBClassifier(**{**p, "random_state": SEED + fold})
            clf = safe_fit(clf, X_all[tr_idx], y_adv[tr_idx],
                           eval_set=[(X_all[va_idx], y_adv[va_idx])],
                           verbose=False)

        oof[va_idx] = predict_best_proba(clf, X_all[va_idx])

    auc = roc_auc_score(y_adv, oof)
    return oof[:X_train.shape[0]], oof[X_train.shape[0]:], float(auc)

def probs_to_weights(p_test, clip_min=0.25, clip_max=4.0):
    p = np.clip(p_test, W_EPS, 1.0 - W_EPS)
    w = p / (1.0 - p)          # with balanced train/test, proportional to p_test/p_train
    w = np.clip(w, clip_min, clip_max)
    w = w / np.mean(w)         # normalize mean weight to 1
    return w.astype(np.float32)

# -----------------------------
# Weighted group calibration (optional)
# -----------------------------
def weighted_group_calibration(train_df, oof_pred, y_true, w_train, test_df, test_pred, n_bins=10):
    bins = build_prior_bins(train_df["prior_ed_cost_5y_usd"].astype(float), n_bins=n_bins)
    tr_bin = pd.cut(train_df["prior_ed_cost_5y_usd"].astype(float), bins=bins, include_lowest=True).astype(str)
    te_bin = pd.cut(test_df["prior_ed_cost_5y_usd"].astype(float), bins=bins, include_lowest=True).astype(str)

    tr_key = (train_df["primary_chronic"].astype(str) + "|" + tr_bin).values
    te_key = (test_df["primary_chronic"].astype(str) + "|" + te_bin).values

    resid = y_true - oof_pred
    # weighted median residual per group
    shift = {}
    for k in np.unique(tr_key):
        m = (tr_key == k)
        shift[k] = weighted_median(resid[m], w_train[m])

    oof_adj = oof_pred + np.array([shift.get(k, 0.0) for k in tr_key], dtype=float)
    te_adj  = test_pred + np.array([shift.get(k, 0.0) for k in te_key], dtype=float)
    return oof_adj, te_adj, shift

# -----------------------------
# MAIN
# -----------------------------
log(f"xgboost version: {getattr(xgb,'__version__','unknown')}")
log(f"PIPELINE_VERSION: {PIPELINE_VERSION}")

train = normalize_cols(pd.read_csv(TRAIN_CSV))
test  = normalize_cols(pd.read_csv(TEST_CSV))
patients = normalize_cols(pd.read_csv(PATIENTS_CSV))

assert RECEIPT_CACHE.exists(), f"Missing: {RECEIPT_CACHE}"
receipts = load(RECEIPT_CACHE)
if not isinstance(receipts, dict) or len(receipts) < 3000:
    raise RuntimeError("receipts_parsed.joblib structure/size looks wrong. Rebuild it first.")

# Features (cached)
need_rebuild = True
if FEAT_CACHE.exists():
    try:
        cache = load(FEAT_CACHE)
        if cache.get("version") == PIPELINE_VERSION:
            train2, test2 = cache["train2"], cache["test2"]
            X_train, X_test = cache["X_train"], cache["X_test"]
            need_rebuild = False
            log(f"Loaded feature cache: {FEAT_CACHE}")
    except Exception:
        need_rebuild = True

if need_rebuild:
    log("Building features (cache miss/version change)...")
    train2, test2, X_train, X_test = build_features(train, test, patients, receipts)
    dump({"version": PIPELINE_VERSION, "train2": train2, "test2": test2, "X_train": X_train, "X_test": X_test}, FEAT_CACHE)
    log(f"Saved feature cache: {FEAT_CACHE}")

y = train2["ed_cost_next3y_usd"].astype(float).values
log(f"X_train shape={X_train.shape}, nnz={X_train.nnz}")
log(f"Target: mean={y.mean():.3f}, p50={np.percentile(y,50):.3f}, p95={np.percentile(y,95):.3f}, max={y.max():.3f}")

# -----------------------------
# Adversarial scores + weights (cached)
# -----------------------------
need_adv = True
if ADV_CACHE.exists():
    try:
        adv = load(ADV_CACHE)
        if adv.get("version") == PIPELINE_VERSION:
            p_train, p_test, adv_auc = adv["p_train"], adv["p_test"], adv["auc"]
            need_adv = False
            log(f"Loaded adversarial cache: {ADV_CACHE}")
    except Exception:
        need_adv = True

if need_adv:
    log("Training adversarial validator (train vs test)...")
    p_train, p_test, adv_auc = adversarial_oof_scores(X_train, X_test)
    dump({"version": PIPELINE_VERSION, "p_train": p_train, "p_test": p_test, "auc": adv_auc}, ADV_CACHE)
    log(f"Saved adversarial cache: {ADV_CACHE}")

log(f"Adversarial AUC (train vs test) = {adv_auc:.4f}  (0.5=no shift; higher=more shift)")
w_train = probs_to_weights(p_train, clip_min=W_CLIP_MIN, clip_max=W_CLIP_MAX)
log(f"Importance weights stats: {sanity_stats(w_train)}")

# Save adv diagnostics
pd.DataFrame({"patient_id": train2["patient_id"].astype(int), "p_test": p_train, "w": w_train}).to_csv(RUN_DIR/"adv_train_scores.csv", index=False)
pd.DataFrame({"patient_id": test2["patient_id"].astype(int), "p_test": p_test}).to_csv(RUN_DIR/"adv_test_scores.csv", index=False)

# -----------------------------
# Regression CV (shift-aware weights)
# Stratify by primary_chronic x prior_cost_decile x adv_score_bin
# -----------------------------
prior_bin = pd.qcut(train2["prior_ed_cost_5y_usd"].astype(float), q=10, duplicates="drop").astype(str)
adv_bin   = pd.qcut(pd.Series(p_train), q=5, duplicates="drop").astype(str)
strat = (train2["primary_chronic"].astype(str) + "|" + prior_bin + "|" + adv_bin).values

skf = StratifiedKFold(n_splits=N_SPLITS_REG, shuffle=True, random_state=SEED)

oof = np.zeros(len(train2), dtype=float)
test_pred = np.zeros(len(test2), dtype=float)

fold_logs = []
log("Starting shift-aware CV training...")

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(train2)), strat), 1):
    t_fold = time.time()
    X_tr, X_va = X_train[tr_idx], X_train[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]
    w_tr, w_va = w_train[tr_idx], w_train[va_idx]

    reg = xgb.XGBRegressor(**{**REG_PARAMS, "random_state": SEED + fold})
    # Fit with train weights; pass eval weights if supported (more test-aligned early stopping)
    eval_set = [(X_tr, y_tr), (X_va, y_va)]
    eval_wts = [w_tr, w_va]
    try:
        reg = safe_fit(reg, X_tr, y_tr, eval_set=eval_set, sample_weight=w_tr, sample_weight_eval_set=eval_wts, verbose=VERBOSE_EVAL)
        mode = "cuda"
    except xgb.core.XGBoostError as e:
        # CPU fallback
        p = dict(REG_PARAMS); p["device"] = "cpu"
        reg = xgb.XGBRegressor(**{**p, "random_state": SEED + fold})
        reg = safe_fit(reg, X_tr, y_tr, eval_set=eval_set, sample_weight=w_tr, sample_weight_eval_set=eval_wts, verbose=VERBOSE_EVAL)
        mode = "cpu"

    pred_tr = np.clip(predict_best(reg, X_tr), 0.0, None)
    pred_va = np.clip(predict_best(reg, X_va), 0.0, None)
    pred_te = np.clip(predict_best(reg, X_test), 0.0, None)

    # Metrics
    train_mae = mean_absolute_error(y_tr, pred_tr)
    val_mae   = mean_absolute_error(y_va, pred_va)
    val_wmae  = weighted_mae(y_va, pred_va, w_va)  # proxy for LB under covariate shift

    oof[va_idx] = pred_va
    test_pred += pred_te / N_SPLITS_REG

    bi = getattr(reg, "best_iteration", None)
    bs = getattr(reg, "best_score", None)

    info = dict(
        fold=fold,
        mode=mode,
        train_mae=float(train_mae),
        val_mae=float(val_mae),
        val_weighted_mae=float(val_wmae),
        best_iteration=None if bi is None else int(bi),
        best_score=None if bs is None else float(bs),
        time_sec=float(time.time() - t_fold),
    )
    fold_logs.append(info)

    log(f"[Fold {fold}/{N_SPLITS_REG}] mode={mode} | TrainMAE={train_mae:.2f} | ValMAE={val_mae:.2f} | "
        f"Val_wMAE={val_wmae:.2f} | best_iter={info['best_iteration']} | time={info['time_sec']:.1f}s")

oof_mae = mean_absolute_error(y, oof)
oof_wmae = weighted_mae(y, oof, w_train)
log(f"OOF MAE (unweighted): {oof_mae:.6f}")
log(f"OOF MAE (importance-weighted proxy): {oof_wmae:.6f}")

# -----------------------------
# Optional weighted calibration (apply only if it improves weighted OOF)
# -----------------------------
test_pred_final = test_pred.copy()
cal_used = False
cal_info = {}

if USE_WEIGHTED_GROUP_CALIBRATION:
    log("Trying weighted group-median residual calibration...")
    oof_cal, test_cal, shift = weighted_group_calibration(train2, oof, y, w_train, test2, test_pred, n_bins=CALIB_BINS)
    oof_mae_cal  = mean_absolute_error(y, oof_cal)
    oof_wmae_cal = weighted_mae(y, oof_cal, w_train)

    log(f"Calibrated OOF MAE: {oof_mae_cal:.6f} (delta {oof_mae - oof_mae_cal:+.6f})")
    log(f"Calibrated OOF wMAE: {oof_wmae_cal:.6f} (delta {oof_wmae - oof_wmae_cal:+.6f})")

    # decision: only use calibration if weighted proxy improves
    if oof_wmae_cal < oof_wmae - 1e-6:
        test_pred_final = np.clip(test_cal, 0.0, None)
        cal_used = True
        cal_info = {"n_groups": len(shift), "oof_mae_cal": float(oof_mae_cal), "oof_wmae_cal": float(oof_wmae_cal)}
        log("Calibration ACCEPTED (improved weighted proxy).")
    else:
        log("Calibration REJECTED (did not improve weighted proxy).")

# -----------------------------
# Save artifacts
# -----------------------------
oof_df = pd.DataFrame({"patient_id": train2["patient_id"].astype(int), "y_true": y, "oof_pred": oof, "w": w_train, "p_test": p_train})
oof_df.to_csv(RUN_DIR/"oof_predictions.csv", index=False)

sub = pd.DataFrame({"patient_id": test2["patient_id"].astype(int), "ed_cost_next3y_usd": np.clip(test_pred_final, 0.0, None)})
sub.to_csv(RUN_DIR/"submission.csv", index=False)
sub.to_csv(SUBMISSION_PATH, index=False)

run_log = dict(
    run_id=RUN_ID,
    pipeline_version=PIPELINE_VERSION,
    xgboost_version=getattr(xgb, "__version__", "unknown"),
    adv_auc=float(adv_auc),
    weight_stats=sanity_stats(w_train),
    feature_shape=[int(X_train.shape[0]), int(X_train.shape[1])],
    feature_nnz=int(X_train.nnz),
    oof_mae=float(oof_mae),
    oof_wmae=float(oof_wmae),
    calibration_used=bool(cal_used),
    calibration_info=cal_info,
    folds=fold_logs,
)
with open(RUN_DIR/"run_log.json", "w", encoding="utf-8") as f:
    json.dump(run_log, f, indent=2)

log(f"Saved run artifacts to: {RUN_DIR}")
log(f"Saved submission -> {SUBMISSION_PATH}")


[06:41:14] xgboost version: 3.0.0
[06:41:14] PIPELINE_VERSION: EDCOST_V2_adv_weighting_2026-02-14
[06:41:15] Building features (cache miss/version change)...
[06:41:15] Saved feature cache: D:\AgentDs\agent_ds_healthcare\cache_iter10\feat_matrix_cache_v2.joblib
[06:41:15] X_train shape=(2000, 65623), nnz=90215
[06:41:15] Target: mean=3908.252, p50=3569.095, p95=7541.847, max=11184.610
[06:41:15] Training adversarial validator (train vs test)...
[06:42:01] Saved adversarial cache: D:\AgentDs\agent_ds_healthcare\cache_iter10\adv_scores_cache_v2.joblib
[06:42:01] Adversarial AUC (train vs test) = 0.5274  (0.5=no shift; higher=more shift)
[06:42:01] Importance weights stats: {'min': 0.2361346185207367, 'max': 3.778153896331787, 'mean': 0.999999995559454, 'p01': 0.2739429384469986, 'p50': 0.9470024108886719, 'p99': 2.8402921867370603}
[06:42:01] Starting shift-aware CV training...
[0]	validation_0-mae:1413.43999	validation_1-mae:1372.90322
[200]	validation_0-mae:284.30608	validation_1-mae:4

# Iteration 3
- 546.2783

In [30]:
# ============================================================
# AgentDS Challenge 2 — V3: Baseline + Residual + Supervised Code-Effect Embeddings
# Key pivot:
#   - Remove high-dimensional hashed code features (overfit source)
#   - Build stable baseline from (primary_chronic x insurance x prior_cost_decile)
#   - Train residual model on low-dim receipt phenotypes + curated high-lift codes
#     + fold-safe supervised code-effect aggregate features.
#
# XGBoost 3.0.0 compatible: early_stopping_rounds + eval_metric in constructor.
# ============================================================

import os, time, json, math, warnings
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from joblib import load, dump
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error

import xgboost as xgb

warnings.filterwarnings("ignore")
np.set_printoptions(suppress=True)

# -----------------------------
# Paths
# -----------------------------
BASE_DIR = Path(r"D:\AgentDs\agent_ds_healthcare")
TRAIN_CSV = BASE_DIR / "ed_cost_train.csv"
TEST_CSV  = BASE_DIR / "ed_cost_test.csv"
PATIENTS_CSV = BASE_DIR / "patients.csv"
CACHE_DIR = BASE_DIR / "cache_iter10"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RECEIPT_CACHE = CACHE_DIR / "receipts_parsed.joblib"
SUBMISSION_PATH = BASE_DIR / "submission_ICHI_V1.csv"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = CACHE_DIR / f"run_v3_{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

def log(msg):
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}] {msg}")

# -----------------------------
# Config
# -----------------------------
SEED = 42
N_SPLITS = 5

# Baseline bins
COST_BINS = 10

# Supervised code effect embedding controls
CODE_MIN_SUPPORT = 15      # minimum patients in fold-train having code to compute effect
SHRINK_ALPHA = 50.0        # shrinkage toward 0: effect = sum_resid / (count + alpha)

# Residual model params (conservative / low-variance)
XGB_RESID_PARAMS = dict(
    n_estimators=5000,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=50,
    subsample=0.75,
    colsample_bytree=0.75,
    reg_lambda=15.0,
    reg_alpha=0.5,
    objective="reg:absoluteerror",
    eval_metric="mae",
    early_stopping_rounds=250,
    tree_method="hist",
    device="cuda",     # will fallback to cpu if needed
    n_jobs=-1,
    verbosity=1,
)

VERBOSE_EVAL = 200

# -----------------------------
# Evidence-driven code sets (from your EDA)
# -----------------------------
SEVERE_PROC = ["31500", "92950", "36556", "36620"]
CRIT_CARE   = ["99291", "99292"]
HI_DIAG     = ["70450", "74177", "84484"]
OBS         = ["G0378"]
ED_LEVELS   = ["99281", "99282", "99283", "99284", "99285"]

CURATED_CODES = SEVERE_PROC + CRIT_CARE + HI_DIAG + OBS + ED_LEVELS

# -----------------------------
# Utilities
# -----------------------------
def normalize_cols(df):
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    return df

def safe_float(x):
    if x is None:
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    s = str(x).strip().replace("$","").replace(",","")
    try:
        return float(s)
    except Exception:
        return np.nan

def gini_from_amounts(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return 0.0
    if np.all(x == 0):
        return 0.0
    x = np.sort(x)
    n = len(x)
    cumx = np.cumsum(x)
    return float((n + 1 - 2 * np.sum(cumx) / (cumx[-1] + 1e-12)) / n)

def entropy_from_counts(counts: Counter) -> float:
    if not counts:
        return 0.0
    v = np.array(list(counts.values()), dtype=float)
    p = v / (v.sum() + 1e-12)
    return float(-(p * np.log(p + 1e-12)).sum())

def code_bucket(code: str) -> str:
    if not isinstance(code, str) or not code:
        return "unk"
    if code.isdigit():
        if code.startswith("99"):
            return "em_99"
        return f"num_{code[0]}"
    return f"alpha_{code[0].upper()}"

def build_prior_bins(prior, n_bins=10):
    q = np.quantile(prior.astype(float), np.linspace(0, 1, n_bins + 1))
    bins = np.unique(q)
    if len(bins) < 3:
        bins = np.unique(np.quantile(prior.astype(float), [0, 0.5, 1.0]))
    return bins

def baseline_group_median(train_df, y, idx_fit, idx_apply, bins):
    fit = train_df.iloc[idx_fit]
    app = train_df.iloc[idx_apply]

    fit_bin = pd.cut(fit["prior_ed_cost_5y_usd"].astype(float), bins=bins, include_lowest=True).astype(str)
    app_bin = pd.cut(app["prior_ed_cost_5y_usd"].astype(float), bins=bins, include_lowest=True).astype(str)

    fit_key = (fit["primary_chronic"].astype(str) + "|" + fit["insurance"].astype(str) + "|" + fit_bin).values
    app_key = (app["primary_chronic"].astype(str) + "|" + app["insurance"].astype(str) + "|" + app_bin).values

    med = pd.Series(y[idx_fit]).groupby(fit_key).median()
    med_dict = med.to_dict()

    # fallbacks
    chronic_med = pd.Series(y[idx_fit]).groupby(fit["primary_chronic"].astype(str).values).median().to_dict()
    global_med = float(np.median(y[idx_fit]))

    out = np.empty(len(idx_apply), dtype=float)
    for i, k in enumerate(app_key):
        if k in med_dict:
            out[i] = med_dict[k]
        else:
            ch = app.iloc[i]["primary_chronic"]
            out[i] = chronic_med.get(str(ch), global_med)
    return out

def extract_codes_and_amounts(obj):
    items = obj.get("items", []) if isinstance(obj, dict) else []
    codes = []
    amounts = []
    for it in items:
        if not isinstance(it, dict):
            continue
        c = str(it.get("code","")).strip()
        a = safe_float(it.get("amount", np.nan))
        if c:
            codes.append(c)
        amounts.append(a)
    return codes, np.array(amounts, dtype=float)

def receipt_phenotype(obj):
    codes, amts = extract_codes_and_amounts(obj)
    total = float(np.nansum(amts))
    if not np.isfinite(total) or total <= 0:
        total = 1e-9

    cc = Counter(codes)
    n_items = int(len(amts))
    n_unique = int(len(cc))

    top_sorted = np.sort(amts[np.isfinite(amts)])[::-1]
    top1 = float(top_sorted[0]) if len(top_sorted) else 0.0
    top3 = float(top_sorted[:3].sum()) if len(top_sorted) >= 3 else float(top_sorted.sum()) if len(top_sorted) else 0.0
    top1_share = top1 / total
    top3_share = top3 / total
    distributed_cost = 1.0 - top3_share

    p = top_sorted / (top_sorted.sum() + 1e-12) if len(top_sorted) else np.array([1.0])
    hhi = float(np.sum(p * p)) if len(p) else 1.0
    gini = gini_from_amounts(amts)

    bucket_entropy = entropy_from_counts(Counter(code_bucket(c) for c in codes))

    # ED lane
    em9928 = [c for c in codes if c in ED_LEVELS]
    n_em_9928x = len(em9928)
    max_em_level = float(max([int(c[-1]) for c in em9928], default=0))

    # critical care
    n_99291 = int(cc.get("99291", 0))
    n_99292 = int(cc.get("99292", 0))
    has_cc  = int((n_99291 + n_99292) > 0)
    cc_time_proxy = float((74 if n_99291 > 0 else 0) + 30 * n_99292)

    # severe procs share
    amt_by_code = Counter()
    for c, a in zip(codes, amts):
        if np.isfinite(a):
            amt_by_code[c] += float(a)
    amt_severe = float(sum(amt_by_code.get(c, 0.0) for c in SEVERE_PROC))
    severe_share = amt_severe / total
    n_severe = int(sum(cc.get(c, 0) for c in SEVERE_PROC))

    # diagnostics
    amt_diag = float(sum(amt_by_code.get(c, 0.0) for c in HI_DIAG))
    diag_share = amt_diag / total
    n_diag = int(sum(cc.get(c, 0) for c in HI_DIAG))

    # severity score
    has_intub = int(cc.get("31500", 0) > 0)
    has_cpr   = int(cc.get("92950", 0) > 0)
    has_cvc   = int(cc.get("36556", 0) > 0)
    has_art   = int(cc.get("36620", 0) > 0)
    has_cth   = int(cc.get("70450", 0) > 0)
    has_ctap  = int(cc.get("74177", 0) > 0)
    has_trop  = int(cc.get("84484", 0) > 0)

    severity_score = 5*has_intub + 5*has_cpr + 4*has_cvc + 4*has_art + 3*has_cc + has_cth + has_ctap + has_trop

    # curated per-code has/cnt/share
    out = {
        "n_items": n_items,
        "n_unique_codes": n_unique,
        "top1_share": float(top1_share),
        "top3_share": float(top3_share),
        "distributed_cost": float(distributed_cost),
        "hhi": float(hhi),
        "gini": float(gini),
        "bucket_entropy": float(bucket_entropy),
        "n_em_9928x": int(n_em_9928x),
        "max_em_level_9928x": float(max_em_level),
        "has_critical_care": int(has_cc),
        "n_99291": int(n_99291),
        "n_99292": int(n_99292),
        "cc_time_proxy": float(cc_time_proxy),
        "n_severe_proc": int(n_severe),
        "severe_proc_share": float(severe_share),
        "n_hi_diag": int(n_diag),
        "hi_diag_share": float(diag_share),
        "severity_score": float(severity_score),
    }

    for c in CURATED_CODES:
        cnt = int(cc.get(c, 0))
        amt = float(amt_by_code.get(c, 0.0))
        out[f"has_{c}"] = int(cnt > 0)
        out[f"cnt_{c}"] = cnt
        out[f"share_{c}"] = float(amt / total)
    return out, cc  # also return counts for code-effect embedding

def build_code_effects(train_pids, resid_by_pid, codes_by_pid, min_support=15, alpha=50.0):
    # Compute shrunk mean residual per code from fold-train only:
    # effect(code) = sum_resid / (count + alpha)
    sum_resid = defaultdict(float)
    cnt = defaultdict(int)

    for pid in train_pids:
        r = float(resid_by_pid[pid])
        codes = codes_by_pid[pid]
        # use unique codes (presence effect), not counts
        for c in codes:
            cnt[c] += 1
            sum_resid[c] += r

    eff = {}
    for c, n in cnt.items():
        if n >= min_support:
            eff[c] = sum_resid[c] / (n + alpha)
    return eff

def code_effect_aggregates(pid_list, codes_by_pid, eff_dict):
    # Aggregate code effects into a few stable features
    rows = []
    for pid in pid_list:
        codes = codes_by_pid[pid]
        vals = [eff_dict.get(c, 0.0) for c in codes]
        vals = np.array(vals, dtype=float) if len(vals) else np.array([0.0], dtype=float)

        pos = vals[vals > 0]
        neg = vals[vals < 0]

        rows.append({
            "ce_sum": float(vals.sum()),
            "ce_mean": float(vals.mean()),
            "ce_max": float(vals.max()),
            "ce_pos_sum": float(pos.sum()) if len(pos) else 0.0,
            "ce_neg_sum": float(neg.sum()) if len(neg) else 0.0,
            "ce_n_codes": float(len(codes)),
        })
    return pd.DataFrame(rows)

def make_model(seed, use_cuda=True):
    p = dict(XGB_RESID_PARAMS)
    p["random_state"] = seed
    if not use_cuda:
        p["device"] = "cpu"
    return xgb.XGBRegressor(**p)

def predict_best(model, X):
    bi = getattr(model, "best_iteration", None)
    if bi is not None:
        try:
            return model.predict(X, iteration_range=(0, int(bi) + 1))
        except TypeError:
            pass
    return model.predict(X)

# -----------------------------
# Load data
# -----------------------------
log(f"xgboost version: {getattr(xgb,'__version__','unknown')}")
train = normalize_cols(pd.read_csv(TRAIN_CSV))
test  = normalize_cols(pd.read_csv(TEST_CSV))
patients = normalize_cols(pd.read_csv(PATIENTS_CSV))

assert RECEIPT_CACHE.exists(), f"Missing {RECEIPT_CACHE}"
receipts = load(RECEIPT_CACHE)
if not isinstance(receipts, dict) or len(receipts) < 3000:
    raise RuntimeError("receipts_parsed.joblib structure/size looks wrong. Rebuild it first.")

# Merge patients
train = train.merge(patients, on="patient_id", how="left")
test  = test.merge(patients, on="patient_id", how="left")

# Precompute receipt features + codes_by_pid (unique code set per patient)
all_df = pd.concat([train.assign(_is_train=1), test.assign(_is_train=0)], axis=0, ignore_index=True)
all_ids = all_df["patient_id"].astype(int).tolist()

feat_rows = []
codes_by_pid = {}
for pid in all_ids:
    obj = receipts.get(int(pid), {"items": []})
    feats, cc = receipt_phenotype(obj)
    feat_rows.append(feats)
    codes_by_pid[int(pid)] = set(cc.keys())

feat_df = pd.DataFrame(feat_rows)
all_df = pd.concat([all_df.reset_index(drop=True), feat_df.reset_index(drop=True)], axis=1)

# Categoricals for residual model
cat_cols = ["primary_chronic", "insurance", "sex", "zip3"]
for c in cat_cols:
    all_df[c] = all_df[c].astype(str)

# Numeric feature list for residual model (exclude raw prior_cost as baseline already uses it)
resid_num_cols = [
    "age",
    "prior_ed_visits_5y",
    "n_items","n_unique_codes","top1_share","top3_share","distributed_cost",
    "hhi","gini","bucket_entropy",
    "n_em_9928x","max_em_level_9928x",
    "has_critical_care","n_99291","n_99292","cc_time_proxy",
    "n_severe_proc","severe_proc_share",
    "n_hi_diag","hi_diag_share",
    "severity_score",
]
# curated code features
for c in CURATED_CODES:
    resid_num_cols += [f"has_{c}", f"cnt_{c}", f"share_{c}"]

# Fill numeric
for c in resid_num_cols:
    all_df[c] = pd.to_numeric(all_df[c], errors="coerce").fillna(0.0).astype(np.float32)

# Fit one-hot encoder on all (no leakage; no y used)
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

X_cat_all = ohe.fit_transform(all_df[cat_cols])

# Split back
train2 = all_df[all_df["_is_train"] == 1].copy().reset_index(drop=True)
test2  = all_df[all_df["_is_train"] == 0].copy().reset_index(drop=True)

X_cat_train = X_cat_all[:len(train2)]
X_cat_test  = X_cat_all[len(train2):]

y = train2["ed_cost_next3y_usd"].astype(float).values

# Global cost bins (no leakage)
bins = build_prior_bins(train2["prior_ed_cost_5y_usd"].values, n_bins=COST_BINS)

# Stratify for CV (stable)
strat = (train2["primary_chronic"].astype(str) + "|" +
         pd.qcut(train2["prior_ed_cost_5y_usd"].astype(float), q=10, duplicates="drop").astype(str)).values
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# OOF containers
oof_pred = np.zeros(len(train2), dtype=float)
oof_base = np.zeros(len(train2), dtype=float)
test_pred_accum = np.zeros(len(test2), dtype=float)
test_base_full = None

fold_logs = []
log(f"Starting V3 CV: baseline + residual | n={len(train2)}")

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(train2)), strat), 1):
    t0 = time.time()

    # fold baseline (no leakage): compute medians using fold-train only
    base_tr = baseline_group_median(train2, y, tr_idx, tr_idx, bins=bins)
    base_va = baseline_group_median(train2, y, tr_idx, va_idx, bins=bins)

    resid_tr = y[tr_idx] - base_tr

    # Build code-effect dictionary from fold-train residuals
    train_pids = train2.loc[tr_idx, "patient_id"].astype(int).tolist()
    resid_by_pid = {int(pid): float(r) for pid, r in zip(train_pids, resid_tr)}
    eff = build_code_effects(train_pids, resid_by_pid, codes_by_pid,
                             min_support=CODE_MIN_SUPPORT, alpha=SHRINK_ALPHA)

    # Code-effect aggregate features (fold-safe)
    tr_pids = train2.loc[tr_idx, "patient_id"].astype(int).tolist()
    va_pids = train2.loc[va_idx, "patient_id"].astype(int).tolist()

    ce_tr = code_effect_aggregates(tr_pids, codes_by_pid, eff).values.astype(np.float32)
    ce_va = code_effect_aggregates(va_pids, codes_by_pid, eff).values.astype(np.float32)

    # Residual model features
    X_num_tr = train2.loc[tr_idx, resid_num_cols].values.astype(np.float32)
    X_num_va = train2.loc[va_idx, resid_num_cols].values.astype(np.float32)

    X_tr = np.hstack([X_num_tr, X_cat_train[tr_idx], ce_tr]).astype(np.float32)
    X_va = np.hstack([X_num_va, X_cat_train[va_idx], ce_va]).astype(np.float32)

    # Train residual model
    model = make_model(SEED + fold, use_cuda=True)
    try:
        model.fit(X_tr, resid_tr,
                  eval_set=[(X_tr, resid_tr), (X_va, y[va_idx] - base_va)],
                  verbose=VERBOSE_EVAL)
        mode = "cuda"
    except xgb.core.XGBoostError as e:
        log(f"[Fold {fold}] GPU failed -> CPU fallback. Error: {repr(e)[:180]}")
        model = make_model(SEED + fold, use_cuda=False)
        model.fit(X_tr, resid_tr,
                  eval_set=[(X_tr, resid_tr), (X_va, y[va_idx] - base_va)],
                  verbose=VERBOSE_EVAL)
        mode = "cpu"

    pred_resid_va = predict_best(model, X_va)
    pred_y_va = np.clip(base_va + pred_resid_va, 0.0, None)

    # Save OOF
    oof_pred[va_idx] = pred_y_va
    oof_base[va_idx] = base_va

    # fold metrics
    mae_base = mean_absolute_error(y[va_idx], base_va)
    mae_full = mean_absolute_error(y[va_idx], pred_y_va)
    info = {
        "fold": fold,
        "mode": mode,
        "mae_baseline": float(mae_base),
        "mae_v3": float(mae_full),
        "improve": float(mae_base - mae_full),
        "best_iteration": int(getattr(model, "best_iteration", -1)) if getattr(model, "best_iteration", None) is not None else None,
        "time_sec": float(time.time() - t0),
        "n_codes_effect": int(len(eff)),
    }
    fold_logs.append(info)
    log(f"[Fold {fold}/{N_SPLITS}] mode={mode} | baseline_MAE={mae_base:.2f} | v3_MAE={mae_full:.2f} | "
        f"improve={mae_base-mae_full:.2f} | best_iter={info['best_iteration']} | "
        f"code_effects={info['n_codes_effect']} | time={info['time_sec']:.1f}s")

# OOF results
oof_mae_base = mean_absolute_error(y, oof_base)
oof_mae_v3 = mean_absolute_error(y, oof_pred)
log(f"OOF MAE baseline-only: {oof_mae_base:.6f}")
log(f"OOF MAE V3 (baseline+residual): {oof_mae_v3:.6f}  (delta {oof_mae_base - oof_mae_v3:+.6f})")

# -----------------------------
# Fit final model on full training and predict test
# -----------------------------
log("Fitting final baseline on full train...")
idx_all = np.arange(len(train2))
base_train_full = baseline_group_median(train2, y, idx_all, idx_all, bins=bins)
base_test_full = baseline_group_median(pd.concat([train2, test2], axis=0, ignore_index=True),
                                       np.concatenate([y, np.zeros(len(test2))]),
                                       np.arange(len(train2)),  # fit idx
                                       np.arange(len(train2), len(train2)+len(test2)),  # apply idx
                                       bins=bins)

resid_full = y - base_train_full

# code effects on full train residual
train_pids_full = train2["patient_id"].astype(int).tolist()
resid_by_pid_full = {int(pid): float(r) for pid, r in zip(train_pids_full, resid_full)}
eff_full = build_code_effects(train_pids_full, resid_by_pid_full, codes_by_pid,
                              min_support=CODE_MIN_SUPPORT, alpha=SHRINK_ALPHA)

ce_train_full = code_effect_aggregates(train_pids_full, codes_by_pid, eff_full).values.astype(np.float32)
test_pids_full = test2["patient_id"].astype(int).tolist()
ce_test_full = code_effect_aggregates(test_pids_full, codes_by_pid, eff_full).values.astype(np.float32)

X_num_train_full = train2[resid_num_cols].values.astype(np.float32)
X_num_test_full  = test2[resid_num_cols].values.astype(np.float32)

X_train_full = np.hstack([X_num_train_full, X_cat_train, ce_train_full]).astype(np.float32)
X_test_full  = np.hstack([X_num_test_full,  X_cat_test,  ce_test_full]).astype(np.float32)

# Choose n_estimators for final fit: use median best_iter from CV if available
best_iters = [f["best_iteration"] for f in fold_logs if f["best_iteration"] is not None and f["best_iteration"] > 0]
final_n_estimators = int(np.median(best_iters)) if best_iters else int(XGB_RESID_PARAMS["n_estimators"])
final_n_estimators = max(200, min(final_n_estimators + 200, XGB_RESID_PARAMS["n_estimators"]))
log(f"Final fit n_estimators = {final_n_estimators} (median_best_iter + 200)")

final_params = dict(XGB_RESID_PARAMS)
final_params["n_estimators"] = final_n_estimators
final_params["early_stopping_rounds"] = None  # final fit no early stop
final_model = xgb.XGBRegressor(**{**final_params, "random_state": SEED + 999})

try:
    final_model.fit(X_train_full, resid_full, verbose=False)
    mode_final = "cuda"
except xgb.core.XGBoostError as e:
    log(f"Final GPU failed -> CPU fallback. Error: {repr(e)[:180]}")
    final_params["device"] = "cpu"
    final_model = xgb.XGBRegressor(**{**final_params, "random_state": SEED + 999})
    final_model.fit(X_train_full, resid_full, verbose=False)
    mode_final = "cpu"

pred_resid_test = final_model.predict(X_test_full)
pred_test = np.clip(base_test_full + pred_resid_test, 0.0, None)

sub = pd.DataFrame({"patient_id": test2["patient_id"].astype(int).values,
                    "ed_cost_next3y_usd": pred_test})
sub.to_csv(SUBMISSION_PATH, index=False)
sub.to_csv(RUN_DIR / "submission.csv", index=False)

# Save OOF + logs
oof_out = pd.DataFrame({
    "patient_id": train2["patient_id"].astype(int),
    "y_true": y,
    "oof_baseline": oof_base,
    "oof_pred": oof_pred
})
oof_out.to_csv(RUN_DIR / "oof_predictions.csv", index=False)

run_log = {
    "run_id": RUN_ID,
    "xgboost_version": getattr(xgb, "__version__", "unknown"),
    "mode_final": mode_final,
    "oof_mae_baseline": float(oof_mae_base),
    "oof_mae_v3": float(oof_mae_v3),
    "folds": fold_logs,
    "code_effect_min_support": CODE_MIN_SUPPORT,
    "code_effect_shrink_alpha": SHRINK_ALPHA,
    "final_n_estimators": final_n_estimators,
}
with open(RUN_DIR / "run_log.json", "w", encoding="utf-8") as f:
    json.dump(run_log, f, indent=2)

log(f"Saved run artifacts: {RUN_DIR}")
log(f"Saved submission -> {SUBMISSION_PATH}")


[14:54:21] xgboost version: 3.0.0
[14:54:21] Starting V3 CV: baseline + residual | n=2000
[0]	validation_0-mae:483.80710	validation_1-mae:542.05221
[200]	validation_0-mae:407.12944	validation_1-mae:533.40803
[347]	validation_0-mae:383.98902	validation_1-mae:536.64686
[14:54:24] [Fold 1/5] mode=cuda | baseline_MAE=542.87 | v3_MAE=529.15 | improve=13.72 | best_iter=98 | code_effects=18 | time=2.5s
[0]	validation_0-mae:485.09723	validation_1-mae:524.23722
[200]	validation_0-mae:406.61182	validation_1-mae:515.21914
[277]	validation_0-mae:393.88378	validation_1-mae:516.37441
[14:54:25] [Fold 2/5] mode=cuda | baseline_MAE=525.42 | v3_MAE=511.84 | improve=13.58 | best_iter=28 | code_effects=18 | time=0.9s
[0]	validation_0-mae:488.53184	validation_1-mae:508.19811
[200]	validation_0-mae:406.09336	validation_1-mae:503.97365
[294]	validation_0-mae:390.67594	validation_1-mae:505.94988
[14:54:26] [Fold 3/5] mode=cuda | baseline_MAE=509.11 | v3_MAE=499.59 | improve=9.52 | best_iter=45 | code_effects

# Iteration 4
- 473.7564

In [34]:
# ============================================================
# EDCOST V3 — Multisource Patient-Level Features (Admissions + Notes + Stays + Vitals) + Receipts
# Metric-aligned: XGBoost reg:absoluteerror (MAE) + group-median residual calibration
# XGBoost 3.0.0 sklearn API compatible: eval_metric + early_stopping_rounds in constructor.
# ============================================================

import os, json, time, math, warnings
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import scipy.sparse as sp

from joblib import load, dump
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

import xgboost as xgb

warnings.filterwarnings("ignore")
np.set_printoptions(suppress=True)

# -----------------------------
# Paths
# -----------------------------
BASE_DIR = Path(r"D:\AgentDs\agent_ds_healthcare")
CACHE_DIR = BASE_DIR / "cache_iter10"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = BASE_DIR / "ed_cost_train.csv"
TEST_CSV  = BASE_DIR / "ed_cost_test.csv"
PATIENTS_CSV = BASE_DIR / "patients.csv"
PDF_DIR = BASE_DIR / "receipts_pdf"                 # not used (we use receipts_parsed.joblib)
RECEIPT_CACHE = CACHE_DIR / "receipts_parsed.joblib"
SUBMISSION_PATH = BASE_DIR / "submission_ICHI_V1.csv"

# extra files (from description.md)
ADM_TR = BASE_DIR / "admissions_train.csv"
ADM_TE = BASE_DIR / "admissions_test.csv"
DISCH_NOTES = BASE_DIR / "discharge_notes.json"
STAY_TR = BASE_DIR / "stays_train.csv"
STAY_TE = BASE_DIR / "stays_test.csv"
VITALS_JSON = BASE_DIR / "vitals_timeseries.json"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = CACHE_DIR / f"run_v3_multi_{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

PIPELINE_VERSION = "EDCOST_V3_MULTISOURCE_2026-02-14"
AUX_CACHE = CACHE_DIR / "patient_aux_features_v3.joblib"
FEAT_CACHE = CACHE_DIR / "feature_matrix_v3.joblib"

def log(msg):
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}] {msg}")

# -----------------------------
# Config
# -----------------------------
SEED = 42
N_SPLITS = 5

# receipt code top-K
TOPK_CODES = 40
CODE_MIN_PATIENT_SUPPORT = 30  # include codes appearing in >= this many patients (train+test)

# Text dims
DISCH_SVD_DIM = 32
VITALSNOTE_SVD_DIM = 16

# XGB params (conservative but strong)
XGB_PARAMS = dict(
    n_estimators=8000,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=10.0,
    reg_alpha=0.2,
    objective="reg:absoluteerror",
    eval_metric="mae",
    early_stopping_rounds=300,      # ctor-based for xgb 3.x
    tree_method="hist",
    device="cuda",
    n_jobs=-1,
    verbosity=1,
)

VERBOSE_EVAL = 200
CALIB_BINS = 10

# -----------------------------
# Helpers
# -----------------------------
def normalize_cols(df):
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    return df

def safe_float(x):
    if x is None:
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    s = str(x).strip().replace("$","").replace(",","")
    try:
        return float(s)
    except Exception:
        return np.nan

def build_prior_bins(train_prior: pd.Series, n_bins: int = 10):
    q = np.quantile(train_prior.values.astype(float), np.linspace(0, 1, n_bins + 1))
    bins = np.unique(q)
    if len(bins) < 3:
        bins = np.unique(np.quantile(train_prior.values.astype(float), [0, 0.5, 1.0]))
    return bins

def predict_best(model, X):
    bi = getattr(model, "best_iteration", None)
    if bi is not None:
        try:
            return model.predict(X, iteration_range=(0, int(bi) + 1))
        except TypeError:
            pass
    return model.predict(X)

def group_median_calibrate(train_df, oof_pred, y_true, test_df, test_pred, n_bins: int = 10):
    bins = build_prior_bins(train_df["prior_ed_cost_5y_usd"].astype(float), n_bins=n_bins)
    tr_bin = pd.cut(train_df["prior_ed_cost_5y_usd"].astype(float), bins=bins, include_lowest=True).astype(str)
    te_bin = pd.cut(test_df["prior_ed_cost_5y_usd"].astype(float), bins=bins, include_lowest=True).astype(str)
    tr_key = (train_df["primary_chronic"].astype(str) + "|" + tr_bin).values
    te_key = (test_df["primary_chronic"].astype(str) + "|" + te_bin).values

    resid = y_true - oof_pred
    shift = pd.Series(resid).groupby(tr_key).median().to_dict()
    oof_adj = oof_pred + np.array([shift.get(k, 0.0) for k in tr_key], dtype=float)
    te_adj  = test_pred + np.array([shift.get(k, 0.0) for k in te_key], dtype=float)
    return oof_adj, te_adj, shift

def tfidf_svd_patient(text_by_pid: pd.Series, n_components: int, prefix: str):
    """
    Fit TF-IDF + SVD on all patients' text (train+test). Return dataframe with SVD comps.
    """
    texts = text_by_pid.fillna("").astype(str).values
    vec = TfidfVectorizer(
        min_df=2,
        max_features=20000,
        ngram_range=(1,2),
        token_pattern=r"(?u)\b\w+\b"
    )
    X = vec.fit_transform(texts)
    k = min(n_components, max(2, X.shape[1]-1)) if X.shape[1] > 2 else 2
    svd = TruncatedSVD(n_components=k, random_state=SEED)
    Z = svd.fit_transform(X)
    cols = [f"{prefix}_svd_{i:02d}" for i in range(Z.shape[1])]
    return pd.DataFrame(Z, columns=cols), {"tfidf_vocab": len(vec.vocabulary_), "svd_dim": Z.shape[1], "explained_var_sum": float(svd.explained_variance_ratio_.sum())}

# -----------------------------
# Receipt features (from receipts_parsed.joblib)
# -----------------------------
SEVERE_PROC = ["31500", "92950", "36556", "36620"]
CRIT_CARE   = ["99291", "99292"]
HI_DIAG     = ["70450", "74177", "84484"]
OBS         = ["G0378"]
ED_LEVELS   = ["99281", "99282", "99283", "99284", "99285"]
CURATED_CODES = SEVERE_PROC + CRIT_CARE + HI_DIAG + OBS + ED_LEVELS

def receipt_patient_features(receipts_dict, patient_ids):
    # pass 1: gather per-patient code support
    support = Counter()
    per_patient = {}
    for pid in patient_ids:
        obj = receipts_dict.get(int(pid), {"items": []})
        items = obj.get("items", [])
        codes = []
        amt_by_code = Counter()
        amts = []
        for it in items:
            if not isinstance(it, dict):
                continue
            c = str(it.get("code","")).strip()
            a = safe_float(it.get("amount", np.nan))
            if c:
                codes.append(c)
            if np.isfinite(a):
                amt_by_code[c] += float(a)
                amts.append(float(a))
        u = set(codes)
        for c in u:
            support[c] += 1
        per_patient[int(pid)] = (Counter(codes), amt_by_code, np.array(amts, dtype=float))

    # select code vocab: top-K by support, but also ensure min support
    code_list = [c for c, n in support.most_common() if n >= CODE_MIN_PATIENT_SUPPORT]
    code_list = code_list[:TOPK_CODES]
    # always include curated high-lift codes even if rare
    for c in CURATED_CODES:
        if c not in code_list:
            code_list.append(c)

    rows = []
    for pid in patient_ids:
        cc, amt_by_code, amts = per_patient[int(pid)]
        total = float(np.nansum(amts))
        if not np.isfinite(total) or total <= 0:
            total = 1e-9

        # concentration / complexity
        top_sorted = np.sort(amts[np.isfinite(amts)])[::-1]
        top1 = float(top_sorted[0]) if len(top_sorted) else 0.0
        top3 = float(top_sorted[:3].sum()) if len(top_sorted) >= 3 else float(top_sorted.sum()) if len(top_sorted) else 0.0
        top1_share = top1 / total
        top3_share = top3 / total
        distributed_cost = 1.0 - top3_share

        # E/M + CC + severe
        n_em = sum(cc.get(c,0) for c in ED_LEVELS)
        n_99291 = int(cc.get("99291", 0))
        n_99292 = int(cc.get("99292", 0))
        has_cc  = int((n_99291 + n_99292) > 0)
        cc_time_proxy = float((74 if n_99291>0 else 0) + 30*n_99292)

        n_severe = int(sum(cc.get(c,0) for c in SEVERE_PROC))
        severe_amt = float(sum(amt_by_code.get(c,0.0) for c in SEVERE_PROC))
        severe_share = severe_amt / total

        n_diag = int(sum(cc.get(c,0) for c in HI_DIAG))
        diag_amt = float(sum(amt_by_code.get(c,0.0) for c in HI_DIAG))
        diag_share = diag_amt / total

        severity_score = (
            5*int(cc.get("31500",0)>0) + 5*int(cc.get("92950",0)>0) +
            4*int(cc.get("36556",0)>0) + 4*int(cc.get("36620",0)>0) +
            3*has_cc + int(cc.get("70450",0)>0) + int(cc.get("74177",0)>0) + int(cc.get("84484",0)>0)
        )

        r = dict(
            patient_id=int(pid),
            rcpt_n_items=int(len(amts)),
            rcpt_n_unique_codes=int(len(cc)),
            rcpt_top1_share=float(top1_share),
            rcpt_top3_share=float(top3_share),
            rcpt_distributed_cost=float(distributed_cost),
            rcpt_n_em_9928x=int(n_em),
            rcpt_has_cc=int(has_cc),
            rcpt_cc_time_proxy=float(cc_time_proxy),
            rcpt_n_severe_proc=int(n_severe),
            rcpt_severe_share=float(severe_share),
            rcpt_n_hi_diag=int(n_diag),
            rcpt_hi_diag_share=float(diag_share),
            rcpt_severity_score=float(severity_score),
        )

        # top code counts + shares
        for c in code_list:
            r[f"code_cnt_{c}"] = float(cc.get(c, 0))
            r[f"code_share_{c}"] = float(amt_by_code.get(c, 0.0) / total)

        rows.append(r)

    return pd.DataFrame(rows), code_list

# -----------------------------
# Admissions + discharge notes features
# -----------------------------
def build_admissions_features():
    if not (ADM_TR.exists() and ADM_TE.exists()):
        log("[AUX] admissions files missing, skipping admissions features.")
        return None

    adm_tr = normalize_cols(pd.read_csv(ADM_TR))
    adm_te = normalize_cols(pd.read_csv(ADM_TE))
    adm = pd.concat([adm_tr.assign(_split="train"), adm_te.assign(_split="test")], axis=0, ignore_index=True)

    # numeric aggregates
    agg = adm.groupby("patient_id").agg(
        adm_n=("admission_id","count"),
        adm_los_mean=("los_days","mean"),
        adm_los_max=("los_days","max"),
        adm_los_std=("los_days","std"),
        adm_aci_mean=("acuity_emergent","mean"),
        adm_charlson_mean=("charlson_band","mean"),
        adm_charlson_max=("charlson_band","max"),
        adm_edvis6m_mean=("ed_visits_6m","mean"),
    ).reset_index()
    agg["adm_los_std"] = agg["adm_los_std"].fillna(0.0)

    # categorical counts
    for col, prefix in [("primary_dx","adm_dx"), ("discharge_weekday","adm_dow")]:
        ctab = pd.crosstab(adm["patient_id"], adm[col]).reset_index()
        ctab.columns = ["patient_id"] + [f"{prefix}_{c}" for c in ctab.columns[1:]]
        agg = agg.merge(ctab, on="patient_id", how="left")

    # discharge notes -> patient text
    note_feat = None
    note_meta = None
    if DISCH_NOTES.exists():
        with open(DISCH_NOTES, "r", encoding="utf-8") as f:
            notes = json.load(f)
        df_notes = pd.DataFrame(notes)
        df_notes = normalize_cols(df_notes)
        # join to admissions to get patient_id
        adm_note = adm.merge(df_notes, on="admission_id", how="left")
        pat_text = adm_note.groupby("patient_id")["note"].apply(lambda s: " ".join([x for x in s.fillna("").astype(str).tolist() if x])).reset_index()
        pat_text = pat_text.set_index("patient_id")["note"]
        Z, meta = tfidf_svd_patient(pat_text.reindex(adm["patient_id"].unique()).fillna(""), DISCH_SVD_DIM, "disch")
        # align rows to patient ids in same order
        pid_order = pd.Index(adm["patient_id"].unique())
        note_feat = pd.DataFrame({"patient_id": pid_order.values})
        note_feat = pd.concat([note_feat.reset_index(drop=True), Z.reset_index(drop=True)], axis=1)
        note_meta = meta
    else:
        log("[AUX] discharge_notes.json missing, skipping discharge note text features.")

    return agg, note_feat, note_meta

# -----------------------------
# Stays + vitals features
# -----------------------------
def build_stays_vitals_features():
    if not (STAY_TR.exists() and STAY_TE.exists()):
        log("[AUX] stays files missing, skipping stays/vitals features.")
        return None

    st_tr = normalize_cols(pd.read_csv(STAY_TR))
    st_te = normalize_cols(pd.read_csv(STAY_TE))
    st = pd.concat([st_tr.assign(_split="train"), st_te.assign(_split="test")], axis=0, ignore_index=True)

    # stays aggregates
    agg = st.groupby("patient_id").agg(
        stay_n=("stay_id","count"),
    ).reset_index()

    for col, prefix in [("unit_type","stay_unit"), ("admission_reason","stay_reason")]:
        ctab = pd.crosstab(st["patient_id"], st[col]).reset_index()
        ctab.columns = ["patient_id"] + [f"{prefix}_{c}" for c in ctab.columns[1:]]
        agg = agg.merge(ctab, on="patient_id", how="left")

    # vitals per stay -> then patient agg
    vit_agg = None
    vit_text_feat = None
    vit_meta = None
    if VITALS_JSON.exists():
        with open(VITALS_JSON, "r", encoding="utf-8") as f:
            vitals = json.load(f)
        rows = []
        note_rows = []
        for obj in vitals:
            sid = int(obj["stay_id"])
            days = obj.get("days", [])
            df = pd.DataFrame(days)
            # numeric summaries per stay
            feat = {"stay_id": sid}
            for col in ["hr","sbp","dbp","temp_c","rr"]:
                if col not in df.columns:
                    feat[f"{col}_mean"] = np.nan
                    feat[f"{col}_std"] = np.nan
                    feat[f"{col}_min"] = np.nan
                    feat[f"{col}_max"] = np.nan
                    feat[f"{col}_slope"] = np.nan
                    continue
                x = pd.to_numeric(df[col], errors="coerce")
                feat[f"{col}_mean"] = float(x.mean())
                feat[f"{col}_std"]  = float(x.std(ddof=0) if np.isfinite(x.std(ddof=0)) else 0.0)
                feat[f"{col}_min"]  = float(x.min())
                feat[f"{col}_max"]  = float(x.max())
                # slope day10 - day1 (if day field exists)
                if "day" in df.columns:
                    dmin = df.loc[df["day"]==1, col]
                    dmax = df.loc[df["day"]==10, col]
                    feat[f"{col}_slope"] = float(pd.to_numeric(dmax, errors="coerce").mean() - pd.to_numeric(dmin, errors="coerce").mean())
                else:
                    feat[f"{col}_slope"] = 0.0
            rows.append(feat)

            # text per stay
            if "note" in df.columns:
                txt = " ".join(df["note"].fillna("").astype(str).tolist())
            else:
                txt = ""
            note_rows.append({"stay_id": sid, "vitals_note": txt})

        vit_stay = pd.DataFrame(rows)
        vit_note = pd.DataFrame(note_rows)

        # join stay -> patient via st
        vit_stay = vit_stay.merge(st[["stay_id","patient_id"]], on="stay_id", how="left")
        vit_note = vit_note.merge(st[["stay_id","patient_id"]], on="stay_id", how="left")

        # patient aggregation of vitals numerics (mean + max)
        agg_mean = vit_stay.groupby("patient_id").mean(numeric_only=True).add_prefix("v_").reset_index()
        agg_max  = vit_stay.groupby("patient_id").max(numeric_only=True).add_prefix("vmax_").reset_index()
        vit_agg = agg_mean.merge(agg_max, on="patient_id", how="left")

        # patient text aggregation of vitals notes -> TFIDF+SVD
        pat_text = vit_note.groupby("patient_id")["vitals_note"].apply(lambda s: " ".join([x for x in s.fillna("").astype(str).tolist() if x])).reset_index()
        pat_text = pat_text.set_index("patient_id")["vitals_note"]

        # fit on all patients that appear in stays
        pid_order = pd.Index(st["patient_id"].unique())
        Z, meta = tfidf_svd_patient(pat_text.reindex(pid_order).fillna(""), VITALSNOTE_SVD_DIM, "vnote")
        vit_text_feat = pd.DataFrame({"patient_id": pid_order.values})
        vit_text_feat = pd.concat([vit_text_feat.reset_index(drop=True), Z.reset_index(drop=True)], axis=1)
        vit_meta = meta
    else:
        log("[AUX] vitals_timeseries.json missing, skipping vitals features.")

    return agg, vit_agg, vit_text_feat, vit_meta

# ============================================================
# MAIN
# ============================================================
log(f"xgboost version: {getattr(xgb,'__version__','unknown')}")
log(f"PIPELINE_VERSION: {PIPELINE_VERSION}")

train = normalize_cols(pd.read_csv(TRAIN_CSV))
test  = normalize_cols(pd.read_csv(TEST_CSV))
patients = normalize_cols(pd.read_csv(PATIENTS_CSV))

assert RECEIPT_CACHE.exists(), f"Missing receipts cache: {RECEIPT_CACHE}"
receipts = load(RECEIPT_CACHE)
if not isinstance(receipts, dict) or len(receipts) < 3000:
    raise RuntimeError("receipts_parsed.joblib structure/size looks wrong. Rebuild it first.")

# concat train/test
train["_is_train"] = 1
test["_is_train"] = 0
all_df = pd.concat([train, test], axis=0, ignore_index=True)
all_df = all_df.merge(patients, on="patient_id", how="left")

# ---- build / load AUX patient features cache
if AUX_CACHE.exists():
    aux = load(AUX_CACHE)
    if aux.get("version") == PIPELINE_VERSION:
        log(f"Loaded AUX cache: {AUX_CACHE}")
        adm_agg = aux.get("adm_agg")
        disch_svd = aux.get("disch_svd")
        stay_agg = aux.get("stay_agg")
        vit_num = aux.get("vit_num")
        vit_svd = aux.get("vit_svd")
    else:
        aux = None
else:
    aux = None

if aux is None:
    log("Building AUX patient-level features (admissions/stays/vitals/notes)...")
    adm_agg, disch_svd, disch_meta = build_admissions_features() if ADM_TR.exists() else (None, None, None)
    stay_agg, vit_num, vit_svd, vit_meta = build_stays_vitals_features() if STAY_TR.exists() else (None, None, None, None)

    dump({
        "version": PIPELINE_VERSION,
        "adm_agg": adm_agg,
        "disch_svd": disch_svd,
        "stay_agg": stay_agg,
        "vit_num": vit_num,
        "vit_svd": vit_svd,
        "disch_meta": disch_meta,
        "vit_meta": vit_meta,
    }, AUX_CACHE)
    log(f"Saved AUX cache: {AUX_CACHE}")

# join AUX to all_df
def safe_merge(base, feat):
    if feat is None:
        return base
    return base.merge(feat, on="patient_id", how="left")

all_df = safe_merge(all_df, adm_agg)
all_df = safe_merge(all_df, disch_svd)
all_df = safe_merge(all_df, stay_agg)
all_df = safe_merge(all_df, vit_num)
all_df = safe_merge(all_df, vit_svd)

# receipts features
log("Building receipt features...")
patient_ids = all_df["patient_id"].astype(int).tolist()
rcpt_feat, code_list = receipt_patient_features(receipts, patient_ids)
all_df = all_df.merge(rcpt_feat, on="patient_id", how="left")
log(f"Receipt top-code features: {len(code_list)} codes")

# fill missing numeric from AUX joins
all_df = all_df.fillna(0.0)

# split
train_df = all_df[all_df["_is_train"] == 1].copy().reset_index(drop=True)
test_df  = all_df[all_df["_is_train"] == 0].copy().reset_index(drop=True)
y = train_df["ed_cost_next3y_usd"].astype(float).values

# feature columns
cat_cols = ["primary_chronic", "sex", "insurance", "zip3"]
for c in cat_cols:
    train_df[c] = train_df[c].astype(str)
    test_df[c]  = test_df[c].astype(str)

drop_cols = {"ed_cost_next3y_usd","_is_train"}
feat_cols = [c for c in train_df.columns if c not in drop_cols]

# separate numeric/cat
num_cols = [c for c in feat_cols if c not in cat_cols and c != "patient_id"]
# convert numeric
for c in num_cols:
    train_df[c] = pd.to_numeric(train_df[c], errors="coerce").fillna(0.0).astype(np.float32)
    test_df[c]  = pd.to_numeric(test_df[c], errors="coerce").fillna(0.0).astype(np.float32)

# one-hot
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=True)

X_cat = ohe.fit_transform(pd.concat([train_df[cat_cols], test_df[cat_cols]], axis=0, ignore_index=True))
X_cat_tr = X_cat[:len(train_df)]
X_cat_te = X_cat[len(train_df):]

X_num_tr = sp.csr_matrix(train_df[num_cols].values)
X_num_te = sp.csr_matrix(test_df[num_cols].values)

X_tr_all = sp.hstack([X_num_tr, X_cat_tr], format="csr")
X_te_all = sp.hstack([X_num_te, X_cat_te], format="csr")

log(f"X shape train={X_tr_all.shape}, nnz={X_tr_all.nnz}")
log(f"Target stats: mean={y.mean():.3f}, p50={np.percentile(y,50):.3f}, p95={np.percentile(y,95):.3f}, max={y.max():.3f}")

# CV split
strat = (train_df["primary_chronic"].astype(str) + "|" +
         pd.qcut(train_df["prior_ed_cost_5y_usd"].astype(float), q=10, duplicates="drop").astype(str)).values
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof = np.zeros(len(train_df), dtype=float)
test_pred = np.zeros(len(test_df), dtype=float)

fold_logs = []
log("Starting CV training (V3 multisource)...")

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(train_df)), strat), 1):
    t0 = time.time()
    Xtr, Xva = X_tr_all[tr_idx], X_tr_all[va_idx]
    ytr, yva = y[tr_idx], y[va_idx]

    model = xgb.XGBRegressor(**{**XGB_PARAMS, "random_state": SEED + fold})
    try:
        model.fit(Xtr, ytr, eval_set=[(Xtr, ytr), (Xva, yva)], verbose=VERBOSE_EVAL)
        mode = "cuda"
    except xgb.core.XGBoostError as e:
        log(f"[Fold {fold}] GPU failed -> CPU fallback. Error: {repr(e)[:180]}")
        p = dict(XGB_PARAMS); p["device"] = "cpu"
        model = xgb.XGBRegressor(**{**p, "random_state": SEED + fold})
        model.fit(Xtr, ytr, eval_set=[(Xtr, ytr), (Xva, yva)], verbose=VERBOSE_EVAL)
        mode = "cpu"

    pred_tr = np.clip(predict_best(model, Xtr), 0.0, None)
    pred_va = np.clip(predict_best(model, Xva), 0.0, None)
    pred_te = np.clip(predict_best(model, X_te_all), 0.0, None)

    tr_mae = mean_absolute_error(ytr, pred_tr)
    va_mae = mean_absolute_error(yva, pred_va)

    oof[va_idx] = pred_va
    test_pred += pred_te / N_SPLITS

    info = dict(
        fold=fold, mode=mode,
        train_mae=float(tr_mae), val_mae=float(va_mae),
        best_iteration=int(getattr(model, "best_iteration", -1)) if getattr(model,"best_iteration",None) is not None else None,
        best_score=float(getattr(model, "best_score", np.nan)) if getattr(model,"best_score",None) is not None else None,
        time_sec=float(time.time()-t0)
    )
    fold_logs.append(info)
    log(f"[Fold {fold}/{N_SPLITS}] mode={mode} | TrainMAE={tr_mae:.2f} | ValMAE={va_mae:.2f} | best_iter={info['best_iteration']} | time={info['time_sec']:.1f}s")

oof_mae = mean_absolute_error(y, oof)
log(f"OOF MAE (raw): {oof_mae:.6f}")

# calibration
oof_cal, test_cal, shift = group_median_calibrate(train_df, oof, y, test_df, test_pred, n_bins=CALIB_BINS)
oof_mae_cal = mean_absolute_error(y, oof_cal)
log(f"OOF MAE (calibrated): {oof_mae_cal:.6f} (delta {oof_mae - oof_mae_cal:+.6f})")

test_pred_final = np.clip(test_cal, 0.0, None)

# Save artifacts
oof_df = pd.DataFrame({"patient_id": train_df["patient_id"].astype(int), "y_true": y, "oof_pred": oof, "oof_pred_cal": oof_cal})
oof_df.to_csv(RUN_DIR / "oof_predictions.csv", index=False)

sub = pd.DataFrame({"patient_id": test_df["patient_id"].astype(int).values, "ed_cost_next3y_usd": test_pred_final})
sub.to_csv(SUBMISSION_PATH, index=False)
sub.to_csv(RUN_DIR / "submission.csv", index=False)

run_log = dict(
    run_id=RUN_ID,
    pipeline_version=PIPELINE_VERSION,
    xgboost_version=getattr(xgb,"__version__","unknown"),
    oof_mae_raw=float(oof_mae),
    oof_mae_cal=float(oof_mae_cal),
    n_features=int(X_tr_all.shape[1]),
    nnz=int(X_tr_all.nnz),
    receipt_code_features=len(code_list),
    folds=fold_logs,
)
with open(RUN_DIR / "run_log.json", "w", encoding="utf-8") as f:
    json.dump(run_log, f, indent=2)

log(f"Saved run artifacts: {RUN_DIR}")
log(f"Saved submission -> {SUBMISSION_PATH}")


[19:12:00] xgboost version: 3.0.0
[19:12:00] PIPELINE_VERSION: EDCOST_V3_MULTISOURCE_2026-02-14
[19:12:00] Building AUX patient-level features (admissions/stays/vitals/notes)...
[19:12:07] Saved AUX cache: D:\AgentDs\agent_ds_healthcare\cache_iter10\patient_aux_features_v3.joblib
[19:12:07] Building receipt features...
[19:12:08] Receipt top-code features: 18 codes
[19:12:08] X shape train=(2000, 195), nnz=207908
[19:12:08] Target stats: mean=3908.252, p50=3569.095, p95=7541.847, max=11184.610
[19:12:08] Starting CV training (V3 multisource)...
[0]	validation_0-mae:1401.38310	validation_1-mae:1387.64869
[200]	validation_0-mae:286.21098	validation_1-mae:457.43628
[400]	validation_0-mae:230.46638	validation_1-mae:454.90287
[600]	validation_0-mae:202.44994	validation_1-mae:454.57743
[678]	validation_0-mae:193.06190	validation_1-mae:454.57177
[19:12:11] [Fold 1/5] mode=cuda | TrainMAE=233.97 | ValMAE=454.26 | best_iter=379 | time=3.8s
[0]	validation_0-mae:1392.04906	validation_1-mae:1422.5

# Iteration 5
- 

## EDA

In [39]:
# =========================
# Iter25 — Verification Cell
# sublinear baseline + utilization/price split + "few but hard" admissions feats
# =========================
from pathlib import Path
import re, math, json, warnings
import numpy as np
import pandas as pd
from joblib import load
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.isotonic import IsotonicRegression
import lightgbm as lgb
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# -------- PATHS (edit if needed) --------
BASE = Path(r"D:\AgentDs\agent_ds_healthcare")
TRAIN_CSV = BASE/"ed_cost_train.csv"
TEST_CSV  = BASE/"ed_cost_test.csv"
PATIENTS_CSV = BASE/"patients.csv"

ADM_TR = BASE/"admissions_train.csv"
ADM_TE = BASE/"admissions_test.csv"

# Use your known-good receipts cache (Iter10 or the fixed one)
PDF_CACHE = BASE/"cache_iter10"/"receipts_parsed.joblib"   # adjust if your fixed cache lives elsewhere

OUT_DIR = BASE/"iter25_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
N_SPLITS = 5
TOL = 1e-2  # ignore tiny float noise

# -------- Helpers --------
def norm_zip3(z):
    if pd.isna(z): return "UNK"
    s = re.sub(r"\D", "", str(z).strip())
    return s.zfill(3) if s else "UNK"

def mae(y, p): return float(mean_absolute_error(y, p))

# -------- Load core tables --------
train = pd.read_csv(TRAIN_CSV)
test  = pd.read_csv(TEST_CSV)
patients = pd.read_csv(PATIENTS_CSV, dtype={"zip3": str})

for d in (train, test, patients):
    d["patient_id"] = d["patient_id"].astype(int)

patients["zip3"] = patients["zip3"].apply(norm_zip3)
patients["insurance"] = patients["insurance"].astype(str).str.lower().str.strip().replace({"nan":"missing"}).fillna("missing")
patients["sex"] = patients["sex"].astype(str).str.strip().replace({"nan":"missing"}).fillna("missing")
patients["zip3_first_digit"] = patients["zip3"].astype(str).str[0].where(patients["zip3"].str.len()>=1, "U")

train = train.merge(patients, on="patient_id", how="left", validate="one_to_one")
test  = test.merge(patients, on="patient_id", how="left", validate="one_to_one")

# -------- Load receipts lineitems from cache --------
obj = load(PDF_CACHE)
if isinstance(obj, dict):
    li = obj.get("lineitems_df", None)
    if li is None:
        # fallback: try common keys
        for k in ["line_items", "lineitems", "items", "df"]:
            if k in obj:
                li = obj[k]
                break
    if li is None:
        raise ValueError("Could not find lineitems_df in receipts_parsed.joblib dict.")
else:
    li = obj

li = li.copy()

# Required columns (try to normalize)
# Expect something like: patient_id, code, qty, line_total, (optional unit_price)
colmap = {c.lower(): c for c in li.columns}
def getc(name):
    return colmap.get(name, None)

pid_c = getc("patient_id")
code_c = getc("code")
qty_c  = getc("qty") or getc("quantity")
lt_c   = getc("line_total") or getc("linetotal") or getc("total")

if pid_c is None or code_c is None or qty_c is None or lt_c is None:
    raise ValueError(f"Lineitems missing required cols. Have: {list(li.columns)}")

li = li.rename(columns={pid_c:"patient_id", code_c:"code", qty_c:"qty", lt_c:"line_total"})
li["patient_id"] = li["patient_id"].astype(int)
li["code"] = li["code"].astype(str).str.strip()
li["qty"] = pd.to_numeric(li["qty"], errors="coerce").fillna(0).astype(int)
li["line_total"] = pd.to_numeric(li["line_total"], errors="coerce").fillna(0.0).astype(float)

# Unit price if not present
li["unit_price"] = li["line_total"] / li["qty"].replace(0, np.nan)
li["unit_price"] = li["unit_price"].replace([np.inf, -np.inf], np.nan)

# -------- Receipt patient-level features --------
# total via sum(line_totals) is the truth-source
r_total = li.groupby("patient_id")["line_total"].sum().rename("pdf_total_sum_items")

r_basic = li.groupby("patient_id").agg(
    pdf_n_line_items=("code","size"),
    pdf_n_unique_codes=("code","nunique"),
    pdf_qty_sum=("qty","sum"),
    pdf_line_total_max=("line_total","max"),
    pdf_line_total_mean=("line_total","mean"),
).reset_index()

r_feat = r_basic.merge(r_total.reset_index(), on="patient_id", how="left")

# utilization vs price split:
# reference price per code = global median unit_price (robust)
ref_price = li.groupby("code")["unit_price"].median()
# fallback for codes where unit_price NaN (qty=0 or weird): use median line_total as proxy
ref_price_fallback = li.groupby("code")["line_total"].median()

def code_ref(c):
    v = ref_price.get(c, np.nan)
    if pd.isna(v): v = ref_price_fallback.get(c, np.nan)
    return v

li["_ref_unit"] = li["code"].map(code_ref).astype(float)
li["_std_line_total"] = li["qty"] * li["_ref_unit"]
li["_std_line_total"] = li["_std_line_total"].fillna(li["line_total"])  # last resort

std_total = li.groupby("patient_id")["_std_line_total"].sum().rename("pdf_standardized_total")
r_feat = r_feat.merge(std_total.reset_index(), on="patient_id", how="left")

r_feat["pdf_price_index"] = r_feat["pdf_total_sum_items"] / (r_feat["pdf_standardized_total"].replace(0, np.nan))
r_feat["pdf_price_index"] = r_feat["pdf_price_index"].replace([np.inf, -np.inf], np.nan).fillna(1.0)
r_feat["log_pdf_standardized_total"] = np.log1p(r_feat["pdf_standardized_total"].clip(lower=0))
r_feat["log_pdf_price_index"] = np.log(r_feat["pdf_price_index"].clip(lower=1e-3))

# HHI concentration on line totals (severity proxy)
tmp = li.merge(r_total.reset_index(), on="patient_id", how="left")
tmp["share"] = tmp["line_total"] / tmp["pdf_total_sum_items"].replace(0, np.nan)
hhi = tmp.groupby("patient_id")["share"].apply(lambda s: float(np.nansum((s.fillna(0).values)**2))).rename("pdf_cost_hhi")
r_feat = r_feat.merge(hhi.reset_index(), on="patient_id", how="left")

# -------- Receipt integrity check --------
chk = train[["patient_id","prior_ed_cost_5y_usd"]].merge(
    r_feat[["patient_id","pdf_total_sum_items"]],
    on="patient_id", how="left"
)
absdiff = (chk["prior_ed_cost_5y_usd"] - chk["pdf_total_sum_items"]).abs()
match_rate = float((absdiff <= TOL).mean())
print(f"[receipt_check] match_rate(|prior - sum_items| <= {TOL}) = {match_rate:.4f}")
print(f"[receipt_check] absdiff percentiles: p50={np.nanpercentile(absdiff,50):.6f}  p95={np.nanpercentile(absdiff,95):.6f}")

# -------- Admissions: "few but hard" patient-level feats --------
adm_tr = pd.read_csv(ADM_TR)
adm_te = pd.read_csv(ADM_TE)
adm = pd.concat([adm_tr.assign(_split="train"), adm_te.assign(_split="test")], axis=0, ignore_index=True)
adm["patient_id"] = adm["patient_id"].astype(int)

# drop label if present
if "readmit_30d" in adm.columns:
    adm = adm.drop(columns=["readmit_30d"])

# patient aggregates
adm_agg = adm.groupby("patient_id").agg(
    adm_n=("admission_id","size") if "admission_id" in adm.columns else ("patient_id","size"),
    adm_charlson_mean=("charlson_band","mean"),
    adm_charlson_max=("charlson_band","max"),
    adm_emergent_rate=("acuity_emergent","mean"),
    adm_los_mean=("los_days","mean"),
    adm_los_max=("los_days","max"),
    adm_edvis6m_mean=("ed_visits_6m","mean"),
    adm_edvis6m_sum=("ed_visits_6m","sum"),
    adm_edvis6m_max=("ed_visits_6m","max"),
).reset_index()

# dx counts (one-hot counts)
if "primary_dx" in adm.columns:
    dx_ct = pd.crosstab(adm["patient_id"], adm["primary_dx"])
    dx_ct.columns = [f"adm_dx_count_{c}" for c in dx_ct.columns]
    dx_ct = dx_ct.reset_index()
    adm_agg = adm_agg.merge(dx_ct, on="patient_id", how="left")

# fill missing admissions feats with 0 (patients w/o admissions rows)
def merge_feats(df):
    out = df.merge(r_feat, on="patient_id", how="left")
    out = out.merge(adm_agg, on="patient_id", how="left")
    for c in out.columns:
        if c.startswith("pdf_") or c.startswith("adm_"):
            if out[c].dtype.kind in "biufc":
                out[c] = out[c].fillna(0.0)
            else:
                out[c] = out[c].fillna("missing")
    return out

train_f = merge_feats(train)
test_f  = merge_feats(test)

# -------- Baselines for CV --------
y = train_f["ed_cost_next3y_usd"].astype(float).values

group_cols = ["primary_chronic","insurance"]

def ratio_baseline_fit_predict(df_tr, df_va):
    eps = 1e-9
    r = (df_tr["ed_cost_next3y_usd"].values / (df_tr["prior_ed_cost_5y_usd"].values + eps))
    tmp = df_tr[group_cols].copy()
    tmp["ratio"] = r
    grp_med = tmp.groupby(group_cols)["ratio"].median()
    global_med = float(np.median(r))
    key_va = list(map(tuple, df_va[group_cols].values))
    ratios = np.array([grp_med.get(k, global_med) for k in key_va], dtype=float)
    pred = df_va["prior_ed_cost_5y_usd"].values * ratios
    return pred

def powerlaw_baseline_fit_predict(df_tr, df_va, min_group=30):
    # log1p(y) = a + b*log1p(prior_cost) + c*log1p(prior_visits)
    global_lr = LinearRegression()
    Xg = np.vstack([
        np.log1p(df_tr["prior_ed_cost_5y_usd"].values),
        np.log1p(df_tr["prior_ed_visits_5y"].values)
    ]).T
    yg = np.log1p(df_tr["ed_cost_next3y_usd"].values)
    global_lr.fit(Xg, yg)

    pred = np.zeros(len(df_va), dtype=float)

    for key, idx_va in df_va.groupby(group_cols).groups.items():
        df_tr_g = df_tr[(df_tr[group_cols[0]]==key[0]) & (df_tr[group_cols[1]]==key[1])]
        if len(df_tr_g) >= min_group:
            lr = LinearRegression()
            X = np.vstack([
                np.log1p(df_tr_g["prior_ed_cost_5y_usd"].values),
                np.log1p(df_tr_g["prior_ed_visits_5y"].values)
            ]).T
            ylog = np.log1p(df_tr_g["ed_cost_next3y_usd"].values)
            lr.fit(X, ylog)
            Xva = np.vstack([
                np.log1p(df_va.loc[idx_va,"prior_ed_cost_5y_usd"].values),
                np.log1p(df_va.loc[idx_va,"prior_ed_visits_5y"].values)
            ]).T
            pred[idx_va] = np.expm1(lr.predict(Xva))
        else:
            Xva = np.vstack([
                np.log1p(df_va.loc[idx_va,"prior_ed_cost_5y_usd"].values),
                np.log1p(df_va.loc[idx_va,"prior_ed_visits_5y"].values)
            ]).T
            pred[idx_va] = np.expm1(global_lr.predict(Xva))

    return np.clip(pred, 0, None)

def isotonic_baseline_fit_predict(df_tr, df_va, min_group=50):
    # per-group isotonic on x=log1p(prior_cost) -> y
    # fallback to global isotonic if group small
    xg = np.log1p(df_tr["prior_ed_cost_5y_usd"].values)
    yg = df_tr["ed_cost_next3y_usd"].values
    iso_g = IsotonicRegression(out_of_bounds="clip")
    iso_g.fit(xg, yg)

    pred = np.zeros(len(df_va), dtype=float)
    for key, idx_va in df_va.groupby(group_cols).groups.items():
        df_tr_g = df_tr[(df_tr[group_cols[0]]==key[0]) & (df_tr[group_cols[1]]==key[1])]
        if len(df_tr_g) >= min_group:
            iso = IsotonicRegression(out_of_bounds="clip")
            iso.fit(np.log1p(df_tr_g["prior_ed_cost_5y_usd"].values), df_tr_g["ed_cost_next3y_usd"].values)
            pred[idx_va] = iso.predict(np.log1p(df_va.loc[idx_va,"prior_ed_cost_5y_usd"].values))
        else:
            pred[idx_va] = iso_g.predict(np.log1p(df_va.loc[idx_va,"prior_ed_cost_5y_usd"].values))
    return np.clip(pred, 0, None)

# very-light LGBM (small feature set)
base_num = [
    "prior_ed_cost_5y_usd","prior_ed_visits_5y","age",
    "log_pdf_standardized_total","log_pdf_price_index","pdf_cost_hhi",
    "adm_charlson_mean","adm_charlson_max","adm_emergent_rate","adm_edvis6m_sum",
]
cat_cols = ["primary_chronic","insurance","sex","zip3","zip3_first_digit"]

# ensure columns exist
for c in base_num:
    if c not in train_f.columns:
        train_f[c] = 0.0
        test_f[c] = 0.0
for c in cat_cols:
    if c not in train_f.columns:
        train_f[c] = "missing"
        test_f[c] = "missing"
    train_f[c] = train_f[c].astype("category")
    test_f[c] = test_f[c].astype("category")

X_cols = base_num + cat_cols

def lgbm_cv_oof(df, y, seed=SEED):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    strat = (df["primary_chronic"].astype(str) + "_" + df["insurance"].astype(str)).values

    oof = np.zeros(len(df), dtype=float)

    params = dict(
        objective="regression_l1",
        n_estimators=20000,
        learning_rate=0.03,
        num_leaves=63,
        min_data_in_leaf=30,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=1.0,
        n_jobs=-1,
        force_col_wise=True,
        verbose=-1,
        random_state=seed,
    )

    for fold,(tr_idx,va_idx) in enumerate(skf.split(df, strat), 1):
        Xtr = df.iloc[tr_idx][X_cols].copy()
        Xva = df.iloc[va_idx][X_cols].copy()
        ytr, yva = y[tr_idx], y[va_idx]

        m = lgb.LGBMRegressor(**params)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="l1",
              callbacks=[lgb.early_stopping(300, verbose=False)])
        oof[va_idx] = m.predict(Xva, num_iteration=m.best_iteration_)

    return oof

# -------- Run CV comparison --------
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
strat = (train_f["primary_chronic"].astype(str) + "_" + train_f["insurance"].astype(str)).values
q95 = float(np.quantile(y, 0.95))

oof_ratio = np.zeros(len(train_f))
oof_pow   = np.zeros(len(train_f))
oof_iso   = np.zeros(len(train_f))

for fold,(tr_idx,va_idx) in enumerate(skf.split(train_f, strat), 1):
    df_tr = train_f.iloc[tr_idx].reset_index(drop=True)
    df_va = train_f.iloc[va_idx].reset_index(drop=False)  # keep original indices
    idx_va = df_va["index"].values

    oof_ratio[idx_va] = ratio_baseline_fit_predict(df_tr, df_va)
    oof_pow[idx_va]   = powerlaw_baseline_fit_predict(df_tr, df_va)
    oof_iso[idx_va]   = isotonic_baseline_fit_predict(df_tr, df_va)

oof_lgb = lgbm_cv_oof(train_f, y)

def report(name, pred):
    overall = mae(y, pred)
    mask = y >= q95
    out_mae = mae(y[mask], pred[mask])
    print(f"[CV] {name}: MAE={overall:.3f} | Outlier(>=p95) MAE={out_mae:.3f}")
    return overall, out_mae

m1 = report("ratio_baseline", oof_ratio)
m2 = report("powerlaw_log_baseline", oof_pow)
m3 = report("isotonic_baseline", oof_iso)
m4 = report("very_light_LGBM", oof_lgb)

# -------- Quick diagnostics plots --------
plt.figure()
plt.scatter(train_f["prior_ed_cost_5y_usd"].values, y, s=8)
plt.xscale("log"); plt.yscale("log")
plt.xlabel("prior_ed_cost_5y_usd (log)"); plt.ylabel("ed_cost_next3y_usd (log)")
plt.title("Shape check: next3y vs prior (log-log)")
plt.tight_layout()
plt.savefig(OUT_DIR/"shape_loglog.png", dpi=160)

plt.figure()
res = y - oof_pow
plt.scatter(train_f["prior_ed_cost_5y_usd"].values, res, s=8)
plt.xscale("log")
plt.axhline(0, linewidth=1)
plt.xlabel("prior_ed_cost_5y_usd (log)"); plt.ylabel("residual (y - powerlaw_pred)")
plt.title("Residual vs prior after power-law baseline")
plt.tight_layout()
plt.savefig(OUT_DIR/"residual_after_powerlaw.png", dpi=160)

print(f"[saved] {OUT_DIR/'shape_loglog.png'}")
print(f"[saved] {OUT_DIR/'residual_after_powerlaw.png'}")


ValueError: Could not find lineitems_df in receipts_parsed.joblib dict.

# New Iter

In [42]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import xgboost

import ed_cost_multimodal_ensemble as m

print("xgboost version:", xgboost.__version__)
print("module:", m.__file__)

DATA_DIR = Path(r"D:\AgentDs\agent_ds_healthcare")   # <-- change if needed
RECEIPTS_DIR = DATA_DIR / "receipts_pdf"
CACHE_DIR = DATA_DIR / "_cache"
CACHE_DIR.mkdir(exist_ok=True)

# -------------------------
# Load base tables
# -------------------------
df_train = pd.read_csv(DATA_DIR / "ed_cost_train.csv")
df_test  = pd.read_csv(DATA_DIR / "ed_cost_test.csv")

y = df_train["ed_cost_next3y_usd"].values
df_train_base = df_train.drop(columns=["ed_cost_next3y_usd"]).copy()
df_test_base  = df_test.copy()

# Optional patients.csv
patients_path = DATA_DIR / "patients.csv"
if patients_path.exists():
    patients = pd.read_csv(patients_path)
    df_train_base = df_train_base.merge(patients, on="patient_id", how="left")
    df_test_base  = df_test_base.merge(patients, on="patient_id", how="left")
    print("[INFO] Loaded patients.csv:", patients.shape)

# Safe base transforms
for df in (df_train_base, df_test_base):
    df["prior_cost_log1p"] = np.log1p(df["prior_ed_cost_5y_usd"])
    df["prior_visits_log1p"] = np.log1p(df["prior_ed_visits_5y"])
    df["prior_cost_per_visit"] = df["prior_ed_cost_5y_usd"] / (df["prior_ed_visits_5y"] + 1e-3)

# -------------------------
# Admissions aggregates
# -------------------------
adm_tr = pd.read_csv(DATA_DIR / "admissions_train.csv")
adm_te = pd.read_csv(DATA_DIR / "admissions_test.csv")
adm_all = pd.concat([adm_tr.drop(columns=["readmit_30d"], errors="ignore"), adm_te], ignore_index=True)
adm_feat = m.build_admissions_aggregates(adm_all)

# -------------------------
# NLP features
# -------------------------
with open(DATA_DIR / "discharge_notes.json", "r") as f:
    discharge_notes = json.load(f)

patient_notes = m.build_patient_notes(adm_all, discharge_notes)
nlp_feat = m.build_nlp_features(patient_notes, cache_path=CACHE_DIR / "nlp_features.joblib")

# -------------------------
# Receipt features (robust to corrupt PDFs in the fixed script)
# -------------------------
all_pids = pd.concat([df_train_base["patient_id"], df_test_base["patient_id"]]).unique()

if RECEIPTS_DIR.exists():
    receipt_feat = m.build_receipt_feature_table(
        receipts_dir=RECEIPTS_DIR,
        patient_ids=all_pids,
        cache_path=CACHE_DIR / "receipt_features.joblib",
        n_jobs=4,  # set to 1 if you get notebook multiprocessing weirdness
    )
else:
    print("[WARN] receipts_pdf folder not found; running without PDF features.")
    receipt_feat = pd.DataFrame({"patient_id": all_pids, "pdf_missing_flag": 1})

# -------------------------
# Merge
# -------------------------
def merge_all(df_base: pd.DataFrame) -> pd.DataFrame:
    out = (
        df_base
        .merge(adm_feat, on="patient_id", how="left")
        .merge(nlp_feat, on="patient_id", how="left")
        .merge(receipt_feat, on="patient_id", how="left")
    )
    if "pdf_total_usd" in out.columns:
        out["pdf_minus_prior_cost"] = out["pdf_total_usd"] - out["prior_ed_cost_5y_usd"]
        out["receipt_mismatch_flag"] = (out["pdf_minus_prior_cost"].abs() > 0.01).astype(int)
    else:
        out["pdf_minus_prior_cost"] = np.nan
        out["receipt_mismatch_flag"] = 0
    return out

df_train_feat = merge_all(df_train_base)
df_test_feat  = merge_all(df_test_base)

# categoricals
cat_cols = [c for c in ["primary_chronic", "insurance", "zip3", "sex", "pdf_insurance", "pdf_zip3"] if c in df_train_feat.columns]
for c in cat_cols:
    df_train_feat[c] = df_train_feat[c].astype("string").fillna("missing")
    df_test_feat[c]  = df_test_feat[c].astype("string").fillna("missing")

# -------------------------
# Train CV ensemble + predict
# -------------------------
cv_res = m.train_cv_ensemble(df_train_feat, y, cat_cols, n_splits=5)
print(f"[OOF] MAE LGBM={cv_res.mae_lgbm:.4f} | XGB={cv_res.mae_xgb:.4f} | CAT={cv_res.mae_cat:.4f}")
print("[OOF] weights:", cv_res.weights)
print("[OOF] best iters:", cv_res.best_iter_lgbm, cv_res.best_iter_xgb, cv_res.best_iter_cat)

pred_test, meta = m.fit_full_models(df_train_feat, y, df_test_feat, cat_cols, cv_res)

sub = pd.DataFrame({"patient_id": df_test_base["patient_id"], "ed_cost_next3y_usd": pred_test})
out_path = DATA_DIR / "submission_ed_cost.csv"
sub.to_csv(out_path, index=False)
print("Wrote:", out_path)
meta


xgboost version: 3.0.0
module: d:\AgentDs\agent_ds_healthcare\ed_cost_multimodal_ensemble.py
[INFO] Loaded patients.csv: (4000, 5)


ValueError: Cannot use median strategy with non-numeric data:
could not convert string to float: ''

# Submission

In [35]:
# 3. Submit Predictions

# Submit predictions to the competition
print("🚀 Submitting predictions...")

try:
    result = client.submit_prediction("Healthcare", 2, "D:/AgentDs/agent_ds_healthcare/submission_ICHI_V1.csv")
    
    if result['success']:
        print("✅ Submission successful!")
        print(f"   📊 Score: {result['score']:.4f}")
        print(f"   📏 Metric: {result['metric_name']}")
        print(f"   ✔️  Validation: {'Passed' if result['validation_passed'] else 'Failed'}")
    else:
        print("❌ Submission failed!")
        print(f"   Error details: {result.get('details', {}).get('validation_errors', 'Unknown error')}")
        
except Exception as e:
    print(f"💥 Submission error: {e}")
    print("🔧 Check your API key and team name are correct!")

print("\n🎯 Next steps:")
print("   1. Try incorporating relevant information outside this table!")
print("   2. Move on to Healthcare Challenge 3!")


🚀 Submitting predictions...
✅ Prediction submitted successfully!
📊 Score: 473.7564 (MAE)
✅ Validation passed
✅ Submission successful!
   📊 Score: 473.7564
   📏 Metric: MAE
   ✔️  Validation: Passed

🎯 Next steps:
   1. Try incorporating relevant information outside this table!
   2. Move on to Healthcare Challenge 3!
